# Patient Question Classification — AMLH Coursework (NLP Dataset C)

Predict a disease label from a patient question: 906 classes, 8,891 training questions,
200 test questions. Three approaches are implemented and compared:

| | Arm | Report section |
|---|---|---|
| **Arm 1** | TF-IDF retrieval + k-NN over a class-blob index | §2.3 |
| **Arm 2** | Fine-tuned Bio_ClinicalBERT, 906-way classification head | §2.4 |
| **Arm 3** | Arm 1 shortlist re-ranked by an LLM (MediPhi-Guidelines) | §2.4 |

This notebook accompanies the report and produces every number in it. Section headers give
the report section they support, so any figure or table can be traced to the cell behind it.

Two rules hold throughout:

1. **`answer` is never an input at prediction time.** Only `question` is. Training-side
   answers may be indexed as class evidence; the test CSV has no `answer` column, and
   `data.load_test` drops it again at the loader.
2. **The test set informs no choice.** Every model, hyperparameter, preprocessing, index
   and prompt decision is made on validation in §2–§4. Test predictions are produced once,
   in §5. The exception is §1.4–§1.6, which read test questions and labels for the
   distributional diagnostics noted there; those select nothing.


## §0.1 Environment

Versions are printed by the run rather than stated. The target runtime is a free Google
Colab **T4** — Arms 2 and 3 need a GPU.


In [ ]:
!pip install -q -U transformers accelerate
!python -m spacy download en_core_web_sm -q

In [ ]:
import importlib
import platform
import sys
import time

NOTEBOOK_START = time.perf_counter()
SECTION_TIMES = {}


def mark(section):
    """Record wall clock at the end of a section; §6 prints the table."""
    elapsed = time.perf_counter() - NOTEBOOK_START
    SECTION_TIMES[section] = elapsed
    print(f"[{section}] cumulative wall clock: {elapsed / 60:.1f} min")


# False: the two expensive sweeps (Arm 2's 24 epochs x 2 encoders, Arm 3's 3 conditions x
#        2 models) are loaded from artefacts/precomputed/. The selection rules still run live.
# True : both sweeps are re-run, adding roughly 4-5 hours on a T4.
RUN_FULL_SELECTION = False

print(f"python           {platform.python_version()}")
for name in ("numpy", "pandas", "sklearn", "scipy", "matplotlib", "seaborn",
             "spacy", "torch", "transformers"):
    try:
        print(f"{name:16s} {importlib.import_module(name).__version__}")
    except ImportError:
        print(f"{name:16s} NOT INSTALLED")

try:
    import torch
    print(f"\ncuda available   {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"gpu              {torch.cuda.get_device_name(0)}")
        print(f"gpu memory       {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
except ImportError:
    print("\ntorch not installed")

print(f"\nRUN_FULL_SELECTION = {RUN_FULL_SELECTION}")

## §0.2 Inputs

Everything read here arrives in **`amlh_submission_inputs.zip`**, submitted alongside this
notebook. Upload it when prompted. The archive holds:

- `data/patient_qa_classification_train.csv` — 8,891 questions over 906 classes
- `data/patient_qa_classification_test.csv` — 200 questions, **no `answer` column**
- `data/db_nhs_qa_classification/*.txt` — the 906 NHS reference documents
- `artefacts/split_fit.csv`, `artefacts/split_val.csv` — the validation split every result
  in the report was computed on (see §1.3)
- `artefacts/precomputed/*.csv` — the recorded sweeps §3 and §4 load when
  `RUN_FULL_SELECTION = False`

Members are stored at project-relative paths, so unzipping in the working directory puts
each file where `config.py` expects it.


In [ ]:
import os
import zipfile
from pathlib import Path

ARCHIVE = "amlh_submission_inputs.zip"

try:
    from google.colab import files  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not Path(ARCHIVE).exists():
    if IN_COLAB:
        from google.colab import files
        files.upload()  # select amlh_submission_inputs.zip
    else:
        raise FileNotFoundError(
            f"{ARCHIVE} not found in {Path.cwd()}. Place it beside the notebook and re-run."
        )

with zipfile.ZipFile(ARCHIVE) as zf:
    zf.extractall(".")
    members = zf.namelist()

print(f"unzipped {len(members)} members into {Path.cwd()}")

In [ ]:
# Check the inputs before anything is computed. A missing NHS document directory does not
# raise on its own: it makes every class document an empty string, which degrades the frozen
# QLAD index to QLA.
import pandas as pd

docs = sorted(Path("data/db_nhs_qa_classification").glob("*.txt"))
assert len(docs) == 906, f"expected 906 NHS documents, found {len(docs)}"

train_check = pd.read_csv("data/patient_qa_classification_train.csv")
assert train_check["disease"].nunique() == 906, train_check["disease"].nunique()

test_check = pd.read_csv("data/patient_qa_classification_test.csv")
assert "answer" not in test_check.columns, "test CSV carries answers -- rebuild the archive"
assert len(test_check) == 200, len(test_check)

for name in ("split_fit.csv", "split_val.csv"):
    assert Path("artefacts", name).is_file(), f"missing artefacts/{name} -- see §1.3"

precomputed = sorted(Path("artefacts/precomputed").glob("*.csv"))

print(f"train            {len(train_check)} rows, {train_check['disease'].nunique()} classes")
print("splits           split_fit.csv, split_val.csv present (shipped, see §1.3)")
print(f"test             {len(test_check)} rows, columns {list(test_check.columns)}  <- no `answer`")
print(f"NHS documents    {len(docs)}")
print(f"precomputed      {len(precomputed)} files: {[p.name for p in precomputed]}")

## §0.3 Source modules

The project keeps its logic in a package so the validation grids and the frozen test run use
the same code. To submit a single file while keeping that structure, the modules below are
written to `src/amlh/` by `%%writefile` and then imported normally.

| Module | Role | Backs report section |
|---|---|---|
| `config.py` | `SEED = 44`, paths, the frozen `Hyperparameters` dataclass | §2.5 |
| `data.py` | loading, integrity audit, the three split designs | §2.1, §2.5 |
| `features.py` | TF-IDF vectoriser, NHS document cleaning, index construction | §2.2, §2.3 |
| `evaluate.py` | accuracy@k, macro-F1, MRR, McNemar's exact test, bootstrap CIs | §3.2 |
| `arm1_tfidf.py` | k-NN retrieval over the index; supervised baselines | §2.3 |
| `arm1_experiments.py` | Arm 1 grids, ablations, within-1-SE selection | §2.3, §4.2 |
| `arm2_bert.py` | BERT fine-tuning, ranked prediction, epoch selection | §2.4 |
| `arm3_llm.py` | shortlists, prompt building, output parsing, Arm 3 selection | §2.4 |
| `results.py` | frozen test run, cross-arm scoring, error analysis | §3.2, §3.3 |
| `eda.py` | label families, sibling homogeneity, novelty calibration | §3.1 |
| `diagrams.py` | workflow diagrams F4 and F5 | §2.3, §2.4 |

In [ ]:
from pathlib import Path

Path("src/amlh").mkdir(parents=True, exist_ok=True)
Path("artefacts").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)
print("src/amlh, artefacts and figures ready")

In [ ]:
%%writefile src/amlh/__init__.py
"""Coursework package for the AMLH patient question classification task."""


In [ ]:
%%writefile src/amlh/config.py
"""Paths and frozen hyperparameters for the pipeline.

Hyperparameter values are ``None`` until tuned on validation in Arms 1-3, then
frozen here for the final test run.
"""

import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import numpy as np

SEED = 44

PROJECT_ROOT = Path(__file__).resolve().parents[2]
DATA_DIR = PROJECT_ROOT / "data"
TRAIN_CSV = DATA_DIR / "patient_qa_classification_train.csv"
TEST_CSV = DATA_DIR / "patient_qa_classification_test.csv"
NHS_DOCS_DIR = DATA_DIR / "db_nhs_qa_classification"
ARTEFACTS_DIR = PROJECT_ROOT / "artefacts"
FIGURES_DIR = PROJECT_ROOT / "figures"

VAL_SIZE = 200  # matches the test set size
VAL_N_CLASSES = 102  # matches the test set class count

# Index-variant options for the Arm 1 class-evidence blob.
IndexVariant = Literal["Q", "QL", "QLA", "QLAD"]


@dataclass(frozen=True)
class Hyperparameters:
    # Arm 1 - TF-IDF / k-NN. Vectoriser, index variant, indexing scheme and k are
    # fixed on validation with a within-1-SE, prefer-simplest rule; where the
    # standard hold-out does not separate the index variants a shift-aware
    # hold-out breaks the tie. ngram_range is re-tuned for the selected variant,
    # whose blobs are reference-document prose rather than short questions.
    ngram_range: tuple[int, int] | None = (1, 2)
    min_df: int | None = 1
    max_df: float | None = 1.0
    sublinear_tf: bool | None = False
    stop_words: str | list[str] | None = None  # ablation outcome
    lemmatise: bool | None = False  # ablation outcome (spaCy en_core_web_sm)
    index_variant: IndexVariant | None = "QLAD"  # question / label / answer / NHS-document blob
    index_scheme: Literal["class_blob", "additive_per_row"] | None = "class_blob"
    k_neighbors: int | None = 1
    # Arm 2 - fine-tuned BERT with a 906-way classification head. The two encoders
    # are trained with identical settings and compared per item on the standard
    # hold-out with McNemar's exact test. max_length follows the question-length
    # percentiles and is not tuned; learning_rate and batch_size are standard
    # fine-tuning defaults held fixed across both encoders; num_epochs is the
    # earliest epoch within 1 SE of the peak validation accuracy in a 24-epoch sweep.
    bert_model_name: str | None = "emilyalsentzer/Bio_ClinicalBERT"
    max_length: int | None = 48
    learning_rate: float | None = 2e-5
    batch_size: int | None = 16
    num_epochs: int | None = 15
    # Arm 3 - LLM selection over the frozen Arm 1 shortlist. Prompt condition then
    # generator are chosen on validation, each with McNemar's exact test: the
    # condition on the primary generator alone, then the generator at that
    # condition. arm3_max_new_tokens is a name-length budget; arm3_cot_max_new_tokens
    # is wider so the chain-of-thought condition has room for reasoning and an answer.
    shortlist_k: int | None = 20
    llm_temperature: float | None = 0.0
    prompt_mode: str | None = "zero_shot"  # zero_shot / few_shot / cot
    n_shots: int | None = 2
    arm3_model_name: str | None = "microsoft/MediPhi-Guidelines"
    arm3_secondary_model_name: str | None = "google/flan-t5-large"
    arm3_max_new_tokens: int | None = 20
    arm3_cot_max_new_tokens: int | None = 128


HYPERPARAMETERS = Hyperparameters()


def set_seed(seed: int = SEED) -> None:
    """Seed every RNG this project touches. Call before every stochastic stage."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

In [ ]:
%%writefile src/amlh/data.py
"""Loading, integrity checks and the validation splits.

`load_test` drops the `answer` column, so no answer is available at prediction
time.
"""

import json
import re
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

from amlh.config import ARTEFACTS_DIR, SEED, TEST_CSV, TRAIN_CSV, VAL_N_CLASSES, VAL_SIZE

EXPECTED_COLUMNS = {"question", "answer", "disease", "reference_url"}
MIN_FIT_SUPPORT = 2  # examples each class must retain for fitting


@dataclass
class SplitResult:
    fit: pd.DataFrame
    val: pd.DataFrame
    val_classes: list[str]


def load_train():
    """Full training DataFrame, including `answer` (permitted as class evidence)."""
    df = pd.read_csv(TRAIN_CSV)
    missing = EXPECTED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"train missing columns: {missing}")
    return df.dropna(subset=["question", "disease"]).reset_index(drop=True)


def load_test():
    """Test DataFrame with the `answer` column dropped."""
    df = pd.read_csv(TEST_CSV)
    missing = {"question", "disease", "reference_url"} - set(df.columns)
    if missing:
        raise ValueError(f"test missing columns: {missing}")
    df = df.dropna(subset=["question", "disease"]).reset_index(drop=True)
    return df.drop(columns=[c for c in df.columns if c == "answer"])


def run_integrity_audit(train, test):
    """Class counts, duplicates, label families and train-test near-duplication."""
    cnt = train.disease.value_counts()
    audit = {
        "n_train": len(train),
        "n_test": len(test),
        "n_classes_train": int(train.disease.nunique()),
        "n_classes_test": int(test.disease.nunique()),
        "test_labels_unseen_in_train": sorted(set(test.disease) - set(train.disease)),
        "class_support": {
            "min": int(cnt.min()),
            "median": float(cnt.median()),
            "mean": round(float(cnt.mean()), 2),
            "max": int(cnt.max()),
            "sd": round(float(cnt.std()), 2),
            "n_singletons": int((cnt == 1).sum()),
            "n_below_5": int((cnt < 5).sum()),
        },
        "exact_dup_question_disease": int(train.duplicated(["question", "disease"]).sum()),
        "duplicate_question_strings": int(train.duplicated(["question"]).sum()),
    }

    # same question string mapped to more than one disease
    g = train[train.duplicated("question", keep=False)].groupby("question").disease.nunique()
    audit["ambiguous_question_strings"] = int((g > 1).sum())

    # label granularity: shared-prefix families
    fam = pd.Series(sorted(train.disease.unique())).str.split("_").str[0]
    fam_counts = fam.value_counts()
    audit["labels_in_a_family"] = int((fam.map(fam_counts) > 1).sum())
    audit["largest_families"] = fam_counts.head(5).to_dict()

    # near-duplication: vectoriser fitted on train questions only, test transformed
    dup_vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(train.question)
    sim = cosine_similarity(dup_vec.transform(test.question), dup_vec.transform(train.question))
    max_sim = sim.max(axis=1)
    audit["near_duplication"] = {
        f"frac_cos_ge_{t}": round(float((max_sim >= t).mean()), 4) for t in (0.95, 0.90, 0.80, 0.70)
    }
    audit["near_duplication"]["median_max_cosine"] = round(float(np.median(max_sim)), 4)

    return audit


def make_validation_split(train, seed=SEED):
    """Class-aware stratified hold-out matching the test set's size and class count.

    Allocates the VAL_SIZE quota across VAL_N_CLASSES up front (via divmod), so no
    truncation is needed and every selected class is guaranteed at least one item.
    """
    rng = np.random.default_rng(seed)
    cnt = train.disease.value_counts()

    base, extra = divmod(VAL_SIZE, VAL_N_CLASSES)

    eligible = cnt[cnt >= base + MIN_FIT_SUPPORT].index.to_numpy()
    if len(eligible) < VAL_N_CLASSES:
        raise ValueError(
            f"only {len(eligible)} classes can supply {base} item(s) "
            f"while retaining {MIN_FIT_SUPPORT}"
        )
    val_classes = rng.choice(eligible, size=VAL_N_CLASSES, replace=False)

    quota = {c: base for c in val_classes}
    bonus_pool = [c for c in val_classes if cnt[c] >= base + 1 + MIN_FIT_SUPPORT]
    if len(bonus_pool) < extra:
        raise ValueError(
            f"only {len(bonus_pool)} classes can supply an extra item; {extra} needed"
        )
    for c in rng.choice(bonus_pool, size=extra, replace=False):
        quota[c] += 1

    val_idx = []
    for c in val_classes:
        pool = train.index[train.disease == c].to_numpy()
        val_idx += list(rng.choice(pool, size=quota[c], replace=False))
    val_idx = list(rng.permutation(val_idx))

    fit = train.drop(index=val_idx)
    val = train.loc[val_idx]

    assert len(val) == VAL_SIZE
    assert val.disease.nunique() == VAL_N_CLASSES
    assert set(val.disease) <= set(fit.disease)
    assert fit.disease.value_counts().reindex(train.disease.unique()).min() >= MIN_FIT_SUPPORT
    assert set(val.index) & set(fit.index) == set()

    return SplitResult(fit=fit, val=val, val_classes=sorted(val_classes))


def _content_words(text):
    """Lowercased words with length > 2 — used to test whether a question
    contains a substantive word from its own disease label."""
    return {w.lower() for w in re.findall(r"\w+", text) if len(w) > 2}


def make_hard_validation_split(train, seed=SEED, n=400):
    """Hold-out drawn only from questions sharing no word with their own label.

    Validation questions in the standard split repeat the phrasing of the
    training questions they were generated with; test questions do not (see
    `eda.sibling_homogeneity`). This split removes that advantage, and is used
    to break ties the standard hold-out cannot resolve. Same quota mechanism as
    `make_validation_split`, but every class with an eligible row takes part.
    """
    rng = np.random.default_rng(seed)

    label_words = {c: _content_words(c.replace("_", " ")) for c in train["disease"].unique()}
    hard_mask = train.apply(
        lambda row: _content_words(row["question"]).isdisjoint(label_words[row["disease"]]), axis=1
    )
    pool = train[hard_mask]

    pool_counts = pool.disease.value_counts()
    full_counts = train.disease.value_counts()

    candidate_classes = sorted(c for c in pool_counts.index if full_counts[c] - MIN_FIT_SUPPORT >= 1)
    n_classes = len(candidate_classes)
    if n_classes == 0:
        raise ValueError("no classes have an eligible (label-word-free) pool member")

    base, extra = divmod(n, n_classes)

    usable = [
        c for c in candidate_classes
        if pool_counts[c] >= base and full_counts[c] - base >= MIN_FIT_SUPPORT
    ]
    if len(usable) < n_classes:
        raise ValueError(
            f"only {len(usable)}/{n_classes} candidate classes can supply {base} item(s) "
            f"while retaining {MIN_FIT_SUPPORT}"
        )

    quota = {c: base for c in candidate_classes}
    bonus_pool = [
        c for c in candidate_classes
        if pool_counts[c] >= base + 1 and full_counts[c] - (base + 1) >= MIN_FIT_SUPPORT
    ]
    if len(bonus_pool) < extra:
        raise ValueError(f"only {len(bonus_pool)} classes can supply an extra item; {extra} needed")
    for c in rng.choice(bonus_pool, size=extra, replace=False):
        quota[c] += 1

    val_idx = []
    for c in candidate_classes:
        if quota[c] == 0:
            continue
        pool_idx = pool.index[pool.disease == c].to_numpy()
        val_idx += list(rng.choice(pool_idx, size=quota[c], replace=False))
    val_idx = list(rng.permutation(val_idx))

    fit = train.drop(index=val_idx)
    val = train.loc[val_idx]

    assert len(val) == n
    assert set(val.index) & set(fit.index) == set()
    assert set(val.disease) <= set(fit.disease)
    assert fit.disease.value_counts().reindex(train.disease.unique()).min() >= MIN_FIT_SUPPORT
    assert all(
        _content_words(row.question).isdisjoint(label_words[row.disease]) for row in val.itertuples()
    )

    return SplitResult(fit=fit, val=val, val_classes=sorted(val.disease.unique()))


def make_random_split(train, seed=SEED):
    """Unstratified hold-out matching the test size, for the robustness check."""
    fit, val = train_test_split(train, test_size=VAL_SIZE, random_state=seed)
    return SplitResult(fit=fit, val=val, val_classes=sorted(val.disease.unique()))


def save_splits(fit, val, audit):
    """Write the splits and audit to artefacts/. `split_fit.csv` keeps `answer`,
    which the QLA and QLAD index variants need."""
    ARTEFACTS_DIR.mkdir(parents=True, exist_ok=True)
    fit.to_csv(ARTEFACTS_DIR / "split_fit.csv", index=False)
    val.to_csv(ARTEFACTS_DIR / "split_val.csv", index=False)
    with open(ARTEFACTS_DIR / "integrity_audit.json", "w") as f:
        json.dump(audit, f, indent=2)


if __name__ == "__main__":
    train = load_train()
    test = load_test()

    audit = run_integrity_audit(train, test)
    split = make_validation_split(train)
    save_splits(split.fit, split.val, audit)

    print(json.dumps(audit, indent=2))
    print(
        f"\nfit={len(split.fit)} | val={len(split.val)} ({split.val.disease.nunique()} classes) "
        f"| test={len(test)}"
    )

    # unstratified split, reported alongside the stratified one
    random_split = make_random_split(train)
    print(
        f"unstratified val: {len(random_split.val)} rows, "
        f"{len(random_split.val_classes)} classes"
    )


In [ ]:
%%writefile src/amlh/features.py
"""Text cleaning and class-evidence index construction for Arm 1.

Builds the texts that `arm1_tfidf.knn_rank` vectorises. Index variants using
`answer` raise on a test frame, which has no such column.
"""

import re
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from amlh.config import NHS_DOCS_DIR

_COMPONENTS = "QLAD"

_DROP_EXACT = {"skip to main content", "- nhs"}
_DROP_PREFIX = ("page last reviewed", "next review due", "credit:")
_URL_RE = re.compile(r"https?://\S+")


def build_vectoriser(**kwargs):
    return TfidfVectorizer(**kwargs)


_FILENAME_INDEX = None


def _filename_index():
    """Map each document filename stem, lowercased, to its path. Built once."""
    global _FILENAME_INDEX
    if _FILENAME_INDEX is None:
        _FILENAME_INDEX = {p.stem.lower(): p for p in NHS_DOCS_DIR.glob("*.txt")}
    return _FILENAME_INDEX


def _strip_boilerplate(raw):
    kept = []
    for line in raw.splitlines():
        s = line.strip()
        if not s:
            continue
        low = s.lower()
        if low in _DROP_EXACT or low.startswith(_DROP_PREFIX):
            continue
        kept.append(_URL_RE.sub("", s).strip())
    return re.sub(r"\s+", " ", " ".join(kept)).strip()


def load_class_doc(disease):
    """Cleaned NHS class document for `disease`, or "" if none exists."""
    path = _filename_index().get(disease.lower())
    if path is None:
        return ""
    return _strip_boilerplate(path.read_text(encoding="utf-8-sig"))


def load_class_doc_raw(disease):
    """Uncleaned NHS class document text for `disease`, or "" if none exists.

    Resolves the filename case-insensitively, like `load_class_doc`. The six
    convention-breaking labels (Bronchitis, Pneumonia, ...) keep their
    capitalised filenames on a case-sensitive filesystem such as Colab's.
    """
    path = _filename_index().get(disease.lower())
    if path is None:
        return ""
    return path.read_text(encoding="utf-8-sig")


def term_class_coverage(diseases):
    """For each token in the NHS documents, how many of `diseases` contain it.
    Uses sklearn's default token pattern, lowercased."""
    token_re = re.compile(r"(?u)\b\w\w+\b")
    counts = {}
    for d in diseases:
        tokens = set(token_re.findall(load_class_doc(d).lower()))
        for t in tokens:
            counts[t] = counts.get(t, 0) + 1
    return counts


def doc_coverage(diseases):
    """NHS class-document coverage over `diseases` (case-insensitive filename match)."""
    idx = _filename_index()
    diseases = list(diseases)
    missing = sorted(d for d in diseases if d.lower() not in idx)
    return {
        "n_total": len(diseases),
        "n_found": len(diseases) - len(missing),
        "missing": missing,
    }


def require_doc_coverage(diseases):
    """Raise if any class in `diseases` has no NHS document.

    `load_class_doc` returns "" for a missing document, so a QLAD index would
    otherwise build as QLA without any error. Called by the index builders
    whenever the variant contains "D".
    """
    coverage = doc_coverage(diseases)
    if coverage["missing"]:
        raise FileNotFoundError(
            f"{len(coverage['missing'])}/{coverage['n_total']} classes have no NHS document under "
            f"{NHS_DOCS_DIR}. A variant containing 'D' cannot be built: the missing classes would "
            f"silently lose their document component. First missing: {coverage['missing'][:5]}"
        )
    return coverage


_NLP = None


def _nlp():
    global _NLP
    if _NLP is None:
        import spacy

        _NLP = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    return _NLP


def lemmatise(texts, drop_stop=True):
    """Lemmatise `texts` with spaCy, batched through `nlp.pipe`."""
    nlp = _nlp()
    out = []
    for doc in nlp.pipe(texts):
        toks = [
            t.lemma_.lower()
            for t in doc
            if not t.is_punct and not t.is_space and (not drop_stop or not t.is_stop)
        ]
        out.append(" ".join(toks))
    return out


def build_index(fit_df, variant):
    """One row per class, joining the requested Q/L/A/D components.

    Components are always concatenated in Q, L, A, D order whatever order
    `variant` lists them in. A variant containing "A" raises `KeyError` on a
    frame with no `answer` column.
    """
    bad = set(variant) - set(_COMPONENTS)
    if bad:
        raise ValueError(f"variant must be subset of {_COMPONENTS}, got unknown chars {bad}")
    if "D" in variant:
        require_doc_coverage(fit_df["disease"].unique())

    texts = []
    labels = []
    for c, group in fit_df.groupby("disease"):
        parts = []
        if "Q" in variant:
            parts.append(" ".join(group["question"]))
        if "L" in variant:
            parts.append(c.replace("_", " "))
        if "A" in variant:
            parts.append(" ".join(group["answer"]))
        if "D" in variant:
            doc = load_class_doc(c)
            if doc:
                parts.append(doc)
        texts.append(" ".join(parts))
        labels.append(c)
    return texts, labels


def build_index_additive(fit_df, variant):
    """One row per training example (Q, A), plus one row per class (L, D).

    L and D do not scale with class size — a class document runs to roughly
    4,000 characters against an eight-word question — so repeating them on
    every example would swamp the index and make the two schemes
    incomparable.
    """
    bad = set(variant) - set(_COMPONENTS)
    if bad:
        raise ValueError(f"variant must be subset of {_COMPONENTS}, got unknown chars {bad}")
    if "D" in variant:
        require_doc_coverage(fit_df["disease"].unique())

    texts = []
    labels = []

    if "Q" in variant or "A" in variant:
        questions = fit_df["question"] if "Q" in variant else None
        answers = fit_df["answer"] if "A" in variant else None
        diseases = fit_df["disease"]
        for i in range(len(fit_df)):
            parts = []
            if questions is not None:
                parts.append(questions.iloc[i])
            if answers is not None:
                parts.append(answers.iloc[i])
            texts.append(" ".join(parts))
            labels.append(diseases.iloc[i])

    if "L" in variant or "D" in variant:
        for c in fit_df["disease"].unique():
            parts = []
            if "L" in variant:
                parts.append(c.replace("_", " "))
            if "D" in variant:
                doc = load_class_doc(c)
                if doc:
                    parts.append(doc)
            texts.append(" ".join(parts))
            labels.append(c)

    return texts, labels


In [ ]:
%%writefile src/amlh/evaluate.py
"""Scoring for ranked disease predictions, and paired system comparison.

`score_ranked` scores one run; `mcnemar_exact` compares two runs over the same
queries; `bootstrap_accuracy_ci` puts a confidence interval on one run.
"""

import math

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from amlh.config import SEED


def accuracy_at_k(ranked, gold, k):
    """Fraction of queries where gold appears in the top-k predicted labels."""
    return sum(g in r[:k] for r, g in zip(ranked, gold)) / len(gold)


def score_ranked(ranked, gold):
    """ranked[i] = query i's predicted labels, best first. gold[i] = true label."""
    top1 = [r[0] if r else None for r in ranked]

    accuracy = accuracy_at_k(ranked, gold, 1)
    acc_at_5 = accuracy_at_k(ranked, gold, 5)
    macro_f1 = f1_score(gold, top1, average="macro", zero_division=0)

    def _rr(r, g):
        return 1.0 / (r.index(g) + 1) if g in r else 0.0

    mrr = sum(_rr(r, g) for r, g in zip(ranked, gold)) / len(gold)

    return {"accuracy": accuracy, "acc_at_5": acc_at_5, "macro_f1": macro_f1, "mrr": mrr}


def mcnemar_exact(pred_a, pred_b, gold):
    """Exact McNemar test on two systems' top-1 predictions over the same items.

    Both systems answer the same queries, so the comparison is paired and the
    items they agree on carry no evidence; the test uses the discordant pairs
    only. The exact binomial tail is used rather than the chi-square
    approximation, which is unreliable below roughly 25 discordant pairs. The
    discordant counts are returned alongside the p-value.
    """
    if not len(pred_a) == len(pred_b) == len(gold):
        raise ValueError("pred_a, pred_b and gold must be query-aligned and the same length")
    if not gold:
        raise ValueError("cannot compare systems over an empty query set")

    correct_a = [p == g for p, g in zip(pred_a, gold)]
    correct_b = [p == g for p, g in zip(pred_b, gold)]
    only_a_correct = sum(a and not b for a, b in zip(correct_a, correct_b))
    only_b_correct = sum(b and not a for a, b in zip(correct_a, correct_b))
    n_discordant = only_a_correct + only_b_correct

    if n_discordant == 0:
        p_value = 1.0
    else:
        lower_tail = sum(
            math.comb(n_discordant, i) for i in range(min(only_a_correct, only_b_correct) + 1)
        )
        p_value = min(1.0, 2 * lower_tail / 2**n_discordant)

    n = len(gold)
    return {
        "n": n,
        "accuracy_a": sum(correct_a) / n,
        "accuracy_b": sum(correct_b) / n,
        "accuracy_diff": (sum(correct_a) - sum(correct_b)) / n,
        "only_a_correct": only_a_correct,
        "only_b_correct": only_b_correct,
        "n_discordant": n_discordant,
        "p_value": p_value,
    }


def bootstrap_accuracy_ci(pred, gold, n_boot=10_000, alpha=0.05, seed=SEED):
    """Percentile bootstrap 95% CI for one system's top-1 accuracy.

    Resamples items, since the quantity estimated is accuracy over the hold-out.
    Uses a local random generator so the surrounding global seed is unaffected.
    If every item is correct or every item is wrong, all resamples give the same
    accuracy and the interval collapses to a point.
    """
    if len(pred) != len(gold):
        raise ValueError("pred and gold must be query-aligned and the same length")
    if not gold:
        raise ValueError("cannot compute a confidence interval over an empty query set")

    n = len(gold)
    correct = np.array([p == g for p, g in zip(pred, gold)], dtype=float)

    rng = np.random.default_rng(seed)
    resample_idx = rng.integers(0, n, size=(n_boot, n))
    boot_accuracies = correct[resample_idx].mean(axis=1)
    ci_low, ci_high = np.quantile(boot_accuracies, [alpha / 2, 1 - alpha / 2])

    return {
        "accuracy": correct.mean(),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "n": n,
        "n_boot": n_boot,
    }


def pairwise_top1_disagreement(top1_by_key):
    """Symmetric matrix counting items whose top-1 prediction differs between
    each pair of keys. All lists must be the same length and in query order."""
    keys = list(top1_by_key)
    matrix = pd.DataFrame(0, index=keys, columns=keys)
    for i, a in enumerate(keys):
        for b in keys[i + 1 :]:
            changed = sum(x != y for x, y in zip(top1_by_key[a], top1_by_key[b]))
            matrix.loc[a, b] = changed
            matrix.loc[b, a] = changed
    return matrix


def accuracy_coverage_curve(ranked, gold, top_sim, thresholds):
    """Top-1 accuracy over queries whose top similarity clears each threshold.
    `accuracy` is NaN where a threshold covers no queries."""
    rows = []
    n = len(gold)
    for t in thresholds:
        kept = [i for i in range(n) if top_sim[i] >= t]
        n_covered = len(kept)
        coverage = n_covered / n
        if n_covered == 0:
            accuracy = math.nan
        else:
            correct = sum(ranked[i][0] == gold[i] if ranked[i] else False for i in kept)
            accuracy = correct / n_covered
        rows.append(
            {"threshold": t, "coverage": coverage, "n_covered": n_covered, "accuracy": accuracy}
        )
    return pd.DataFrame(rows)


In [ ]:
%%writefile src/amlh/arm1_tfidf.py
"""Arm 1 retrieval: TF-IDF over the class-evidence index, k nearest
neighbours by cosine similarity, and a similarity-weighted vote over the
neighbours' labels. Also holds the supervised TF-IDF baselines compared
against retrieval in the same arm.
"""

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.svm import LinearSVC

from amlh.config import SEED
from amlh.features import build_vectoriser


def knn_rank(index_texts, index_labels, query_texts, k, vec_kwargs, depth=None):
    """Fit TF-IDF on `index_texts`, transform `query_texts`, and rank labels per
    query by summed cosine similarity over the k nearest neighbours.

    Returns (ranked labels per query, top similarity per query).

    At the frozen `k_neighbors=1` the ranking holds a single label, which leaves
    nothing for Top-5, MRR or an Arm 3 shortlist. Passing `depth` retrieves
    further neighbours and appends any new labels *below* the unchanged head, so
    the top-1 prediction is the same either way.
    """
    n_retrieve = min(k, len(index_texts))
    if depth is not None:
        n_retrieve = min(max(k, depth), len(index_texts))

    vec = build_vectoriser(**vec_kwargs)
    X_index = vec.fit_transform(index_texts)
    X_query = vec.transform(query_texts)

    nn = NearestNeighbors(n_neighbors=n_retrieve, metric="cosine")
    nn.fit(X_index)
    distances, neighbour_idx = nn.kneighbors(X_query)
    similarities = 1.0 - distances

    labels = np.asarray(index_labels)
    ranked = []
    top_sim = []
    for row_sims, row_idx in zip(similarities, neighbour_idx):
        vote = {}
        for lab, sim in zip(labels[row_idx[:k]], row_sims[:k]):
            vote[lab] = vote.get(lab, 0.0) + sim
        head = sorted(vote, key=vote.get, reverse=True)

        if depth is not None:
            tail_sim = {}
            for lab, sim in zip(labels[row_idx[k:]], row_sims[k:]):
                if lab in vote:
                    continue
                if sim > tail_sim.get(lab, -1.0):
                    tail_sim[lab] = sim
            head = head + sorted(tail_sim, key=tail_sim.get, reverse=True)

        ranked.append(head)
        top_sim.append(float(row_sims[0]))

    return ranked, top_sim


_MODELS = {
    "svc": lambda seed: LinearSVC(random_state=seed),
    "logreg": lambda seed: LogisticRegression(random_state=seed, max_iter=1000),
    "random_forest": lambda seed: RandomForestClassifier(random_state=seed),
}


def linear_rank(train_texts, train_labels, query_texts, vec_kwargs, model, seed=SEED):
    """Fit TF-IDF and a supervised classifier on the training texts, then rank
    classes per query by decision score, highest first. `model` is one of
    "svc", "logreg", "random_forest".
    """
    if model not in _MODELS:
        raise ValueError(f"model must be one of {sorted(_MODELS)}, got {model!r}")

    vec = build_vectoriser(**vec_kwargs)
    X_train = vec.fit_transform(train_texts)
    X_query = vec.transform(query_texts)

    clf = _MODELS[model](seed)
    clf.fit(X_train, train_labels)

    if hasattr(clf, "predict_proba"):
        scores = clf.predict_proba(X_query)
    else:
        scores = clf.decision_function(X_query)

    classes = clf.classes_
    ranked = []
    top_score = []
    for row in scores:
        order = np.argsort(row)[::-1]
        ranked.append([classes[i] for i in order])
        top_score.append(float(row[order[0]]))

    return ranked, top_score


In [ ]:
%%writefile src/amlh/arm1_experiments.py
"""Grid and ablation loops for Arm 1.

The indexing and scoring themselves live in `features`, `arm1_tfidf` and
`evaluate`; this module only sweeps them over hyperparameters, index variants
and indexing schemes.
"""

import math
from itertools import product

import pandas as pd

from amlh import arm1_tfidf, data, features
from amlh.evaluate import accuracy_at_k, score_ranked


def _score_row(ranked, gold, extra):
    return {**extra, **score_ranked(ranked, gold)}


def run_vectoriser_grid(
    fit_df,
    val_df,
    variant,
    ngram_ranges,
    sublinear_tf_opts,
    min_dfs,
    stop_words_opts,
    ks,
):
    """Sweep vectoriser settings and k over a class-blob index. The index does
    not depend on k, so it is built once per vectoriser combination."""
    gold = val_df["disease"].tolist()
    queries = val_df["question"].tolist()
    rows = []
    for ngram_range, sublinear_tf, min_df, stop_words in product(
        ngram_ranges, sublinear_tf_opts, min_dfs, stop_words_opts
    ):
        vec_kwargs = {
            "ngram_range": ngram_range,
            "sublinear_tf": sublinear_tf,
            "min_df": min_df,
            "stop_words": stop_words,
        }
        index_texts, index_labels = features.build_index(fit_df, variant)
        for k in ks:
            ranked, _ = arm1_tfidf.knn_rank(index_texts, index_labels, queries, k, vec_kwargs)
            rows.append(_score_row(ranked, gold, {**vec_kwargs, "k": k}))
    return pd.DataFrame(rows)


def select_within_one_se(grid_df, n_val=200):
    """Of the configurations within 1 SE of the best validation accuracy, return
    the simplest: lowest k, then unigrams, then no stop-word list, then
    min_df=1. Avoids reading a raw argmax off a noisy 200-item hold-out."""
    best_acc = grid_df["accuracy"].max()
    se = math.sqrt(best_acc * (1 - best_acc) / n_val)
    within = grid_df[grid_df["accuracy"] >= best_acc - se].copy()
    within["_ngram_penalty"] = within["ngram_range"].apply(lambda ng: 0 if tuple(ng) == (1, 1) else 1)
    within["_stopword_penalty"] = within["stop_words"].apply(lambda s: 0 if s is None else 1)
    within = within.sort_values(
        by=["k", "_ngram_penalty", "_stopword_penalty", "min_df"],
        ascending=[True, True, True, True],
    )
    return within.iloc[0].drop(["_ngram_penalty", "_stopword_penalty"])


def run_preprocessing_ablation(fit_df, val_df, vec_kwargs, k):
    """Compare raw text against spaCy lemmatisation with and without stop-word
    removal, on the "Q" index. Each lemmatised version is computed once."""
    gold = val_df["disease"].tolist()
    raw_queries = val_df["question"].tolist()

    lemma_stop_fit_q = features.lemmatise(fit_df["question"].tolist(), drop_stop=True)
    lemma_only_fit_q = features.lemmatise(fit_df["question"].tolist(), drop_stop=False)
    lemma_stop_val_q = features.lemmatise(raw_queries, drop_stop=True)
    lemma_only_val_q = features.lemmatise(raw_queries, drop_stop=False)

    rows = []

    index_texts, index_labels = features.build_index(fit_df, "Q")
    ranked, _ = arm1_tfidf.knn_rank(index_texts, index_labels, raw_queries, k, vec_kwargs)
    rows.append(_score_row(ranked, gold, {"preprocessing": "raw"}))

    for name, fit_q, val_q in (
        ("lemma_stop", lemma_stop_fit_q, lemma_stop_val_q),
        ("lemma_only", lemma_only_fit_q, lemma_only_val_q),
    ):
        fit_lemma = fit_df.copy()
        fit_lemma["question"] = fit_q
        index_texts, index_labels = features.build_index(fit_lemma, "Q")
        ranked, _ = arm1_tfidf.knn_rank(index_texts, index_labels, val_q, k, vec_kwargs)
        rows.append(_score_row(ranked, gold, {"preprocessing": name}))

    return pd.DataFrame(rows)


def run_index_variant_comparison(fit_df, val_df, variants, vec_kwargs, k):
    gold = val_df["disease"].tolist()
    queries = val_df["question"].tolist()
    rows = []
    for variant in variants:
        index_texts, index_labels = features.build_index(fit_df, variant)
        ranked, _ = arm1_tfidf.knn_rank(index_texts, index_labels, queries, k, vec_kwargs)
        rows.append(_score_row(ranked, gold, {"variant": variant}))
    return pd.DataFrame(rows)


def run_indexing_scheme_comparison(fit_df, val_df, variant, vec_kwargs, k):
    gold = val_df["disease"].tolist()
    queries = val_df["question"].tolist()
    rows = []
    for scheme, builder in (
        ("class_blob", features.build_index),
        ("additive_per_row", features.build_index_additive),
    ):
        index_texts, index_labels = builder(fit_df, variant)
        ranked, _ = arm1_tfidf.knn_rank(index_texts, index_labels, queries, k, vec_kwargs)
        rows.append(_score_row(ranked, gold, {"scheme": scheme, "n_index_rows": len(index_texts)}))
    return pd.DataFrame(rows)


def run_variant_scheme_grid(
    fit_df,
    val_df,
    variants,
    vec_kwargs,
    k,
    schemes=("class_blob", "additive_per_row"),
):
    """Score every (scheme, variant) combination on one fit/evaluation pair."""
    builders = {"class_blob": features.build_index, "additive_per_row": features.build_index_additive}
    gold = val_df["disease"].tolist()
    queries = val_df["question"].tolist()
    rows = []
    for scheme in schemes:
        builder = builders[scheme]
        for variant in variants:
            index_texts, index_labels = builder(fit_df, variant)
            ranked, _ = arm1_tfidf.knn_rank(index_texts, index_labels, queries, k, vec_kwargs)
            rows.append(_score_row(ranked, gold, {"scheme": scheme, "variant": variant}))
    return pd.DataFrame(rows)


def run_split_robustness(train_df, configs, variant="Q", seeds=(42, 43, 44)):
    """Re-run the top vectoriser configurations under an unstratified split, one
    row per (config_rank, seed). The fit/validation composition differs from the
    stratified grid, so only the ranking across `configs` is comparable."""
    rows = []
    for seed in seeds:
        split = data.make_random_split(train_df, seed=seed)
        gold = split.val["disease"].tolist()
        queries = split.val["question"].tolist()
        index_texts, index_labels = features.build_index(split.fit, variant)
        for rank, cfg in enumerate(configs, start=1):
            vec_kwargs = {
                "ngram_range": cfg["ngram_range"],
                "sublinear_tf": cfg["sublinear_tf"],
                "min_df": cfg["min_df"],
                "stop_words": cfg["stop_words"],
            }
            ranked, _ = arm1_tfidf.knn_rank(index_texts, index_labels, queries, cfg["k"], vec_kwargs)
            rows.append(
                _score_row(ranked, gold, {"config_rank": rank, "seed": seed, **vec_kwargs, "k": cfg["k"]})
            )
    return pd.DataFrame(rows)


def frozen_ranking(fit_df, val_df, hp, depth=None):
    """Run `knn_rank` at the frozen Arm 1 hyperparameters over `val_df`.

    `depth=None` reproduces the ranking the grids above report; a `depth`
    extends it without changing rank 1. The only place `hp`'s vectoriser, index
    and k fields are read, so both paths read them identically."""
    vec_kwargs = {
        "ngram_range": hp.ngram_range,
        "sublinear_tf": hp.sublinear_tf,
        "min_df": hp.min_df,
        "stop_words": hp.stop_words,
    }
    build_fn = features.build_index if hp.index_scheme == "class_blob" else features.build_index_additive
    index_texts, index_labels = build_fn(fit_df, hp.index_variant)
    return arm1_tfidf.knn_rank(
        index_texts, index_labels, val_df["question"].tolist(), hp.k_neighbors, vec_kwargs, depth=depth
    )


def build_val_predictions(fit_df, val_df, hp, depth):
    """Per-item Arm 1 predictions, ranked `depth` labels deep, in `val_df` order.

    McNemar and the error analysis need per-item output, which the aggregate
    grids do not record. Rows with fewer than `depth` labels are left blank
    rather than padded."""
    ranked, top_sim = frozen_ranking(fit_df, val_df, hp, depth=depth)
    rows = []
    for question, gold, r, sim in zip(val_df["question"], val_df["disease"], ranked, top_sim):
        row = {"question": question, "gold": gold, "pred": r[0]}
        for i in range(depth):
            row[f"top_{i + 1}"] = r[i] if i < len(r) else None
        row["top_sim"] = sim
        rows.append(row)
    return pd.DataFrame(rows)


def shortlist_ceiling(predictions_df, ks):
    """Accuracy@k over a `build_val_predictions` frame's top_1..top_N columns.
    This is the ceiling Arm 3 selects against: it cannot beat the rate at which
    the gold label appears in the shortlist at all."""
    top_cols = [f"top_{i}" for i in range(1, max(ks) + 1)]
    ranked = [[lab for lab in row if isinstance(lab, str)] for row in predictions_df[top_cols].values.tolist()]
    gold = predictions_df["gold"].tolist()
    return pd.DataFrame({"k": ks, "accuracy": [accuracy_at_k(ranked, gold, k) for k in ks]})


def run_supervised_baselines(fit_df, val_df, vec_kwargs):
    """TF-IDF with LinearSVC, LogisticRegression and RandomForestClassifier,
    trained on individual questions rather than on class blobs."""
    gold = val_df["disease"].tolist()
    queries = val_df["question"].tolist()
    train_texts = fit_df["question"].tolist()
    train_labels = fit_df["disease"].tolist()
    rows = []
    for model in ("svc", "logreg", "random_forest"):
        ranked, _ = arm1_tfidf.linear_rank(train_texts, train_labels, queries, vec_kwargs, model)
        rows.append(_score_row(ranked, gold, {"model": model}))
    return pd.DataFrame(rows)


In [ ]:
%%writefile src/amlh/arm2_bert.py
"""Fine-tuning for Arm 2 (BERT classifier).

`question` -> WordPiece -> encoder -> 906-way linear head -> argmax, using the
manual epoch loop from the NLP3 practical (transformers and torch only, no
Trainer). Functions here return frames and tuples; the notebook prints and plots.
"""

import math
import time

import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from transformers import AutoTokenizer, BertForSequenceClassification

from amlh import config
from amlh.evaluate import score_ranked


def encode_labels(train_df):
    """Label-to-id maps over all disease labels, sorted so ids are stable.

    Called on the full training set rather than a split, so the output layer
    covers every class the test run could need to predict."""
    labels = sorted(train_df["disease"].unique())
    label_to_id = {label: i for i, label in enumerate(labels)}
    id_to_label = {i: label for label, i in label_to_id.items()}
    return label_to_id, id_to_label


def tokenise(texts, tokeniser, max_length):
    """Pad/truncate to a fixed length so all batches share tensor shape."""
    return tokeniser(
        texts,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )


def truncation_rate(texts, tokeniser, max_length):
    """Fraction of texts whose untruncated WordPiece length exceeds max_length."""
    lengths = [len(ids) for ids in tokeniser(texts, truncation=False)["input_ids"]]
    return sum(n > max_length for n in lengths) / len(lengths)


def make_loader(encodings, labels, batch_size, shuffle, seed=None):
    """Wrap the encodings in a DataLoader, as in the NLP3 practical. Shuffling
    uses an explicit Generator so each call is reproducible on its own."""
    dataset = TensorDataset(
        encodings["input_ids"], encodings["attention_mask"], torch.tensor(labels, dtype=torch.long)
    )
    if shuffle:
        if seed is None:
            raise ValueError("seed is required when shuffle=True")
        generator = torch.Generator().manual_seed(seed)
        sampler = RandomSampler(dataset, generator=generator)
    else:
        sampler = SequentialSampler(dataset)
    return DataLoader(dataset, sampler=sampler, batch_size=batch_size, num_workers=0)


def _run_epoch(model, loader, optimizer, device, train):
    """One pass over `loader`, training or evaluating. Both pass `labels=` so the
    evaluation pass also returns a loss for the per-epoch loss curves."""
    model.train() if train else model.eval()
    total_loss = 0.0
    n = 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for input_ids, attention_mask, labels in loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            if train:
                model.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            n += len(labels)
    return total_loss / n


def predict_ranked(
    model,
    tokeniser,
    questions,
    id_to_label,
    max_length,
    batch_size,
    device,
    top_k=5,
):
    """Ranked label predictions, best first, in the order of `questions`.

    `top_k=None` ranks every label, which is what MRR needs: a truncated ranking
    scores MRR@k instead, and would not be comparable with Arm 1.
    """
    encodings = tokenise(questions, tokeniser, max_length)
    dummy_labels = [0] * len(questions)
    loader = make_loader(encodings, dummy_labels, batch_size, shuffle=False)

    model.eval()
    ranked = []
    k = len(id_to_label) if top_k is None else min(top_k, len(id_to_label))
    with torch.no_grad():
        for input_ids, attention_mask, _ in loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            top_ids = torch.topk(logits, k=k, dim=1).indices.cpu().tolist()
            ranked.extend([[id_to_label[i] for i in row] for row in top_ids])
    return ranked


def train_model(
    fit_df,
    val_df,
    label_to_id,
    model_name,
    lr,
    batch_size,
    epochs,
    max_length,
    seed,
    model=None,
    tokeniser=None,
    on_epoch_end=None,
):
    """Manual AdamW training loop, following the NLP3 practical.

    Returns (best state dict, history frame, rankings per epoch). The history
    frame has one row per epoch: epoch, train_loss, val_loss, val_accuracy, and
    the best epoch is the highest val_accuracy, earliest on ties.

    The per-epoch validation rankings are kept because they were computed anyway
    to score each epoch. They let per-item predictions be recovered at whichever
    epoch is selected afterwards, without retraining.

    `on_epoch_end`, if given, is called with each epoch's row so the notebook can
    print progress."""
    missing = (set(fit_df["disease"]) | set(val_df["disease"])) - set(label_to_id)
    if missing:
        raise ValueError(f"{len(missing)} labels absent from label_to_id: {sorted(missing)[:5]}")

    config.set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if tokeniser is None:
        tokeniser = AutoTokenizer.from_pretrained(model_name)
    if model is None:
        model = BertForSequenceClassification.from_pretrained(
            model_name, num_labels=len(label_to_id)
        )
    model = model.to(device)

    fit_labels = fit_df["disease"].map(label_to_id).tolist()
    val_labels = val_df["disease"].map(label_to_id).tolist()
    fit_encodings = tokenise(fit_df["question"].tolist(), tokeniser, max_length)
    val_encodings = tokenise(val_df["question"].tolist(), tokeniser, max_length)

    fit_loader = make_loader(fit_encodings, fit_labels, batch_size, shuffle=True, seed=seed)
    val_loader = make_loader(val_encodings, val_labels, batch_size, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr)

    id_to_label = {i: label for label, i in label_to_id.items()}
    val_gold = val_df["disease"].tolist()

    rows = []
    ranked_by_epoch = {}
    best_state = None
    best_val_accuracy = -1.0
    for epoch in range(epochs):
        train_loss = _run_epoch(model, fit_loader, optimizer, device, train=True)
        val_loss = _run_epoch(model, val_loader, optimizer, device, train=False)
        ranked = predict_ranked(
            model, tokeniser, val_df["question"].tolist(), id_to_label, max_length, batch_size, device
        )
        val_accuracy = score_ranked(ranked, val_gold)["accuracy"]
        row = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_accuracy": val_accuracy}
        rows.append(row)
        if on_epoch_end is not None:
            on_epoch_end(row)
        ranked_by_epoch[epoch] = ranked
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return best_state, pd.DataFrame(rows), ranked_by_epoch


def measure_run(fn, *args, **kwargs):
    """Run `fn`, returning (result, wall_clock_s, peak_memory_mb). Peak memory is
    nan rather than 0 when CUDA is unavailable."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    wall_clock_s = time.perf_counter() - start
    if torch.cuda.is_available():
        peak_memory_mb = torch.cuda.max_memory_allocated() / 1e6
    else:
        peak_memory_mb = float("nan")
    return result, wall_clock_s, peak_memory_mb


def run_model_ablation(
    fit_df,
    val_df,
    label_to_id,
    model_names,
    lr,
    batch_size,
    epochs,
    max_length,
    seed,
    on_epoch_end=None,
):
    """Train each encoder with identical hyperparameters, re-seeding before each.

    Returns a summary row per encoder, each encoder's history frame, and each
    encoder's per-epoch validation rankings, so the two can be compared per item
    at whichever epoch is selected without retraining."""
    summary_rows = []
    histories = {}
    ranked_by_epoch_by_model = {}
    for model_name in model_names:
        config.set_seed(seed)
        epoch_callback = (
            None if on_epoch_end is None else (lambda row, model_name=model_name: on_epoch_end(model_name, row))
        )
        (best_state, history_df, ranked_by_epoch), wall_clock_s, peak_memory_mb = measure_run(
            train_model,
            fit_df,
            val_df,
            label_to_id,
            model_name,
            lr,
            batch_size,
            epochs,
            max_length,
            seed,
            on_epoch_end=epoch_callback,
        )
        best_row = history_df.loc[history_df["val_accuracy"].idxmax()]
        histories[model_name] = history_df
        ranked_by_epoch_by_model[model_name] = ranked_by_epoch
        summary_rows.append(
            {
                "model_name": model_name,
                "best_epoch": int(best_row["epoch"]),
                "val_accuracy": best_row["val_accuracy"],
                "val_loss": best_row["val_loss"],
                "wall_clock_s": wall_clock_s,
                "peak_memory_mb": peak_memory_mb,
            }
        )
    return pd.DataFrame(summary_rows), histories, ranked_by_epoch_by_model


def select_best_epoch_within_one_se(history_df, n_val=200):
    """Of the epochs within 1 SE of the best validation accuracy, return the
    earliest. Avoids reading a raw argmax off a noisy 200-item hold-out."""
    best_acc = history_df["val_accuracy"].max()
    se = math.sqrt(best_acc * (1 - best_acc) / n_val)
    within = history_df[history_df["val_accuracy"] >= best_acc - se]
    return within.sort_values("epoch").iloc[0]


In [ ]:
%%writefile src/amlh/arm3_llm.py
"""Arm 3: LLM selection over an Arm 1 shortlist.

Frozen Arm 1 retrieval produces a shortlist of candidate labels. Arm 3 formats
it into one of three prompt conditions, sends the prompt to the generator,
matches the reply back to a shortlist label by name, and falls back to Arm 1's
top-1 when no candidate matches.

This follows Task 4 of the NLP4 practical, which prompts an LLM to pick the most
relevant diagnosis from a candidate list: the generator is called through a
`pipe(prompt, max_new_tokens=...)` closure, prompts are chat messages carrying
the practical's system persona and its "Respond with only the diagnosis name."
instruction, and the reply is matched by name rather than by a candidate number.

Two changes from the practical, both reported in §2.4:

1. The practical picks a random diagnosis when the reply matches nothing. Arm 3
   returns Arm 1's top-1 instead, which is the prediction the system would have
   made without the LLM, so the fallback adds no noise of its own.
2. `flatten_messages` joins the roles with a blank line where the practical's
   API branch joins them directly.
"""

import re
import time
from dataclasses import dataclass

import pandas as pd

from amlh import arm1_experiments, evaluate
from amlh.config import HYPERPARAMETERS, SEED

# The practical's diagnosis-selection system prompt.
SYSTEM_PROMPT = "You are a helpful assistant trained to identify relevant medical question topics."

# Closing instruction per condition. The `cot` wording carries no placeholder
# token, which an earlier version had and the models copied out verbatim instead
# of answering.
_CLOSERS = {
    "zero_shot": "Respond with only the diagnosis name.",
    "few_shot": "Respond with only the diagnosis name.",
    "cot": (
        "Think step by step about which diagnosis the question is asking about, "
        "then end your reply with 'Final answer:' followed by the diagnosis name."
    ),
}


@dataclass(frozen=True)
class PromptExample:
    question: str
    label: str


def prettify_label(label):
    """Replace underscores so labels read as ordinary disease names."""
    return label.replace("_", " ")


def selected_model_name(hp=HYPERPARAMETERS):
    return hp.arm3_model_name or "microsoft/MediPhi-Guidelines"


def secondary_model_name(hp=HYPERPARAMETERS):
    return hp.arm3_secondary_model_name or "google/flan-t5-large"


def shortlist_k(hp=HYPERPARAMETERS):
    if hp.shortlist_k is None:
        raise ValueError("shortlist_k must be frozen in config.py before Arm 3 runs")
    return hp.shortlist_k


def n_shots(hp=HYPERPARAMETERS):
    if hp.n_shots is None:
        return 0
    return hp.n_shots


def llm_temperature(hp=HYPERPARAMETERS):
    if hp.llm_temperature is None:
        raise ValueError("llm_temperature must be frozen in config.py before Arm 3 runs")
    return hp.llm_temperature


def max_new_tokens_for(mode, hp=HYPERPARAMETERS):
    """Generation budget for `mode`. `cot` needs room for reasoning as well as an
    answer; the others need only a name, for which the practical allows 20
    tokens."""
    if mode == "cot":
        return hp.arm3_cot_max_new_tokens or 128
    return hp.arm3_max_new_tokens or 20


def build_shortlist_ranking(fit_df, val_df, hp=HYPERPARAMETERS, depth=None):
    """Shortlist ranking from the frozen Arm 1 configuration, so Arm 3 never
    re-tunes the retriever it is given."""
    return arm1_experiments.frozen_ranking(fit_df, val_df, hp, depth=depth)


def assert_reproduces_arm1(shortlist_rankings, arm1_predictions):
    """Check the shortlist's top-1 against Arm 1's saved per-item predictions.

    Rank 1 identifies the index that produced it, so this catches a shortlist
    built from anything other than the frozen configuration before any prompt is
    sent.
    """
    top1 = [ranking[0] for ranking in shortlist_rankings]
    expected = arm1_predictions["top_1"].tolist()
    if len(top1) != len(expected):
        raise ValueError(
            f"shortlist has {len(top1)} items but arm1_val_predictions.csv has {len(expected)}; "
            "these must be the same validation split in the same order"
        )
    mismatches = [i for i, (a, b) in enumerate(zip(top1, expected)) if a != b]
    if mismatches:
        raise ValueError(
            f"shortlist top-1 disagrees with Arm 1 on {len(mismatches)}/{len(expected)} items "
            f"(first at index {mismatches[0]}: got {top1[mismatches[0]]!r}, "
            f"expected {expected[mismatches[0]]!r}). The frozen retrieval configuration is not "
            "the one that produced arm1_val_predictions.csv — check that data/db_nhs_qa_classification "
            "is present and that config.py has not drifted."
        )
    return {"n": len(top1), "top1_matches_arm1": True}


def build_examples(fit_df, n, seed=SEED):
    """Deterministically sample compact few-shot exemplars from the fit split only."""
    if n <= 0:
        return []
    sample = fit_df.sample(n=min(n, len(fit_df)), random_state=seed).reset_index(drop=True)
    return [PromptExample(question=row.question, label=row.disease) for row in sample.itertuples()]


def candidate_names(shortlist):
    """Candidate labels as the practical renders them: a comma-joined name list."""
    return ", ".join(prettify_label(label) for label in shortlist)


def build_prompt(question, shortlist, mode, examples=None):
    """Build one chat-message prompt for the requested condition, following the
    practical's diagnosis-selection prompt with the candidate list narrowed from
    every disease to the Arm 1 shortlist."""
    if mode not in _CLOSERS:
        raise ValueError(f"unknown prompt mode: {mode!r}")
    examples = examples or []

    parts = [
        "Given the question below, which one of the following diagnoses is most relevant?",
        "",
        "Diagnoses:",
        candidate_names(shortlist),
        "",
    ]

    if examples:
        parts.append("Worked examples:")
        for example in examples:
            parts.append(f"Question:\n{example.question}")
            parts.append(f"Diagnosis:\n{prettify_label(example.label)}")
            parts.append("")

    parts.append(f"Question:\n{question}")
    parts.append("")
    parts.append(_CLOSERS[mode])

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "\n".join(parts).strip()},
    ]


def flatten_messages(messages):
    """Chat messages as one string, for tokenisers with no chat template. Joined
    on a blank line so the persona does not run into the instruction."""
    return "\n\n".join(message["content"] for message in messages)


def build_prompts_for_condition(val_df, shortlist_rankings, mode, examples=None):
    """Return one chat prompt per validation question, aligned to `val_df`."""
    prompts = []
    for question, shortlist in zip(val_df["question"].tolist(), shortlist_rankings):
        prompts.append(build_prompt(question, shortlist, mode, examples=examples))
    return prompts


def prompt_token_lengths(prompts, tokeniser, max_length=512):
    """Measure prompt lengths and the truncation rate at the model encoder limit."""
    texts = [flatten_messages(p) if isinstance(p, list) else p for p in prompts]
    lengths = [len(tokeniser(text, truncation=False)["input_ids"]) for text in texts]
    trunc_rate = sum(length > max_length for length in lengths) / len(lengths) if lengths else 0.0
    series = pd.Series(lengths, dtype="int64") if lengths else pd.Series(dtype="int64")
    return {
        "lengths": lengths,
        "truncation_rate": trunc_rate,
        "min": int(series.min()) if lengths else 0,
        "median": float(series.median()) if lengths else 0.0,
        "mean": float(series.mean()) if lengths else 0.0,
        "p95": float(series.quantile(0.95)) if lengths else 0.0,
        "max": int(series.max()) if lengths else 0,
        "n": len(lengths),
    }


# matched per line, so the last marker in a chain of reasoning wins
_FINAL_ANSWER_RE = re.compile(r"final\s*answer\s*[:\-]?\s*(.*)", re.IGNORECASE)
_NON_ALNUM_RE = re.compile(r"[^a-z0-9]+")


def _normalise(text):
    """Casefold and reduce to single-spaced alphanumerics for name matching."""
    return _NON_ALNUM_RE.sub(" ", text.lower()).strip()


def parse_diagnosis_name(raw_output, shortlist):
    """Match a generated diagnosis name back to a shortlist label.

    Exact match first, as in the practical's membership test. Models often wrap
    the name in a sentence, so a containment pass follows, taking the longest
    match so "leukaemia" cannot shadow "acute myeloid leukaemia". Returns None
    when nothing matches.
    """
    if not raw_output:
        return None
    text = raw_output.strip()
    markers = list(_FINAL_ANSWER_RE.finditer(text))
    if markers:
        text = markers[-1].group(1)

    normalised_output = _normalise(text)
    if not normalised_output:
        return None

    candidates = [(label, _normalise(prettify_label(label))) for label in shortlist]
    for label, normalised_label in candidates:
        if normalised_label == normalised_output:
            return label

    best_label, best_length = None, 0
    for label, normalised_label in candidates:
        if normalised_label and normalised_label in normalised_output and len(normalised_label) > best_length:
            best_label, best_length = label, len(normalised_label)
    return best_label


def parse_model_output(raw_output, shortlist):
    """Map a generation to a shortlist label, or fall back to Arm 1's top-1.

    Returns (label, fallback_fired, matched_label, matched_rank), where
    `matched_rank` is the shortlist position chosen and so shows whether the LLM
    reordered the retriever at all.
    """
    matched = parse_diagnosis_name(raw_output, shortlist)
    if matched is None:
        return shortlist[0], True, None, None
    return matched, False, matched, shortlist.index(matched) + 1


def load_generator(model_name=None, device=None, hp=HYPERPARAMETERS):
    """Load a generator and return (tokenizer, model, pipe).

    `pipe` has the practical's signature and return shape, so the call site
    reads the same. Flan-T5 has no chat template and takes flattened text;
    MediPhi goes through `apply_chat_template`, as in the practical.
    """
    import torch
    from transformers import AutoConfig, AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer

    name = model_name or selected_model_name(hp)
    config = AutoConfig.from_pretrained(name)
    is_encoder_decoder = bool(getattr(config, "is_encoder_decoder", False))

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(name)
    dtype = torch.float16 if str(device).startswith("cuda") else torch.float32
    loader = AutoModelForSeq2SeqLM if is_encoder_decoder else AutoModelForCausalLM
    model = loader.from_pretrained(name, dtype=dtype).to(device)
    model.eval()

    def pipe(
        prompt,
        max_new_tokens=100,  # maximum number of new tokens to generate (excluding the input prompt)
        temperature=0.0,  # controls randomness, with do_sample=False generation is deterministic
        return_full_text=False,  # if True, return both the prompt and generated text
        do_sample=False,  # if False, use greedy decoding
        clean_up_tokenization_spaces=False,  # whether to remove tokenisation artefacts
    ):
        # prompt is expected to be [{"role": "system", ...}, {"role": "user", ...}]
        if is_encoder_decoder:
            inputs = tokenizer(flatten_messages(prompt), return_tensors="pt", truncation=True)
        else:
            inputs = tokenizer.apply_chat_template(
                prompt,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=do_sample,
            )

        # An encoder-decoder's output holds only the generated tokens; a causal
        # model's replays the prompt first and has to be sliced off.
        if is_encoder_decoder or return_full_text:
            generated_ids = outputs[0]
        else:
            generated_ids = outputs[0][inputs["input_ids"].shape[1] :]

        generated_text = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=clean_up_tokenization_spaces,
        )
        return [{"generated_text": generated_text.strip()}]

    return tokenizer, model, pipe


def run_condition(
    fit_df,
    val_df,
    hp=HYPERPARAMETERS,
    mode="zero_shot",
    pipe=None,
    examples=None,
    shortlist_depth=None,
    model_name=None,
    shortlist_rankings=None,
    top_sim=None,
):
    """Run one prompt condition over the validation split.

    Returns the per-item frame, a metrics dict, and the chat prompts used. The
    caller decides whether to persist them. Passing `shortlist_rankings` reuses
    a ranking already built rather than re-running retrieval per condition.
    """
    if pipe is None:
        raise ValueError("a `pipe` callable is required — pass load_generator(...)[2] or a fake")
    shortlist_depth = shortlist_depth or shortlist_k(hp)
    if shortlist_rankings is None:
        shortlist_rankings, top_sim = build_shortlist_ranking(fit_df, val_df, hp, depth=shortlist_depth)
    if top_sim is None:
        top_sim = [float("nan")] * len(shortlist_rankings)

    prompts = build_prompts_for_condition(val_df, shortlist_rankings, mode, examples=examples)
    budget = max_new_tokens_for(mode, hp)
    temperature = llm_temperature(hp)

    rows = []
    start = time.perf_counter()
    for item_idx, (question, gold, shortlist, prompt, sim) in enumerate(
        zip(val_df["question"], val_df["disease"], shortlist_rankings, prompts, top_sim)
    ):
        raw_output = pipe(prompt, max_new_tokens=budget, temperature=temperature)[0]["generated_text"]
        pred, fallback_fired, matched_label, matched_rank = parse_model_output(raw_output, shortlist)
        gold_rank = shortlist.index(gold) + 1 if gold in shortlist else None
        rows.append(
            {
                "condition": mode,
                "model_name": model_name or selected_model_name(hp),
                "item_idx": item_idx,
                "question": question,
                "gold": gold,
                "arm1_pred": shortlist[0],
                "pred": pred,
                "parsed_label": matched_label,
                "parsed_rank": matched_rank,
                "fallback_fired": fallback_fired,
                "gold_rank": gold_rank,
                "top_sim": sim,
                "raw_output": raw_output,
                "prompt": flatten_messages(prompt),
            }
        )

    pred_df = pd.DataFrame(rows)
    elapsed = time.perf_counter() - start
    metrics = summarise_condition(pred_df, elapsed)
    return pred_df, metrics, prompts


def summarise_condition(pred_df, wall_clock_sec):
    """Condition summary: accuracy, fallback rate, CI and wall clock.

    `arm1_accuracy` is the shortlist's own top-1 on the same items, which is the
    baseline the LLM has to beat.
    """
    pred = pred_df["pred"].tolist()
    gold = pred_df["gold"].tolist()
    arm1 = pred_df["arm1_pred"].tolist()
    ci = evaluate.bootstrap_accuracy_ci(pred, gold, seed=SEED)
    accuracy = sum(p == g for p, g in zip(pred, gold)) / len(gold)
    arm1_accuracy = sum(p == g for p, g in zip(arm1, gold)) / len(gold)
    non_fallback = pred_df[~pred_df["fallback_fired"]]
    return {
        "condition": pred_df["condition"].iloc[0] if len(pred_df) else None,
        "model_name": pred_df["model_name"].iloc[0] if len(pred_df) else None,
        "n": len(pred_df),
        "accuracy": accuracy,
        "arm1_accuracy": arm1_accuracy,
        "accuracy_minus_arm1": accuracy - arm1_accuracy,
        "fallback_rate": float(pred_df["fallback_fired"].mean()),
        "agreement_with_arm1": float((pred_df["pred"] == pred_df["arm1_pred"]).mean()),
        "moved_off_rank1_rate": float((non_fallback["parsed_rank"] > 1).mean()) if len(non_fallback) else 0.0,
        "ci_low": ci["ci_low"],
        "ci_high": ci["ci_high"],
        "wall_clock_sec": wall_clock_sec,
    }


def pairwise_condition_mcnemar(condition_frames):
    """Pairwise McNemar comparisons over matched validation items."""
    rows = []
    modes = sorted(condition_frames)
    for i, a in enumerate(modes):
        for b in modes[i + 1 :]:
            frame_a = condition_frames[a]
            frame_b = condition_frames[b]
            result = evaluate.mcnemar_exact(frame_a["pred"].tolist(), frame_b["pred"].tolist(), frame_a["gold"].tolist())
            rows.append({"condition_a": a, "condition_b": b, **result})
    return pd.DataFrame(rows)


def condition_vs_arm1_mcnemar(condition_frames):
    """McNemar of each condition against the Arm 1 top-1 it was given. Paired by
    construction, since both answer the same items."""
    rows = []
    for mode in sorted(condition_frames):
        frame = condition_frames[mode]
        result = evaluate.mcnemar_exact(frame["pred"].tolist(), frame["arm1_pred"].tolist(), frame["gold"].tolist())
        rows.append({"system_a": f"arm3_{mode}", "system_b": "arm1_top1", **result})
    return pd.DataFrame(rows)


def select_prompt_mode(condition_metrics, mcnemar_df):
    """Select the prompt mode, returning (mode, tie_break_fired).

    If no condition beats both others at p < 0.05 the comparison is reported as
    unresolved and `zero_shot` is kept as the simplest prompt. This can retain a
    lower-scoring condition.
    """
    wins = {mode: set() for mode in condition_metrics["condition"]}
    for row in mcnemar_df.itertuples(index=False):
        if row.p_value >= 0.05:
            continue
        if row.accuracy_a > row.accuracy_b:
            wins[row.condition_a].add(row.condition_b)
        elif row.accuracy_b > row.accuracy_a:
            wins[row.condition_b].add(row.condition_a)

    dominating = [mode for mode, beaten in wins.items() if len(beaten) == len(condition_metrics) - 1]
    if len(dominating) == 1:
        return dominating[0], False
    return "zero_shot", True


def model_mcnemar(primary_frame, secondary_frame):
    """McNemar of the two Arm 3 generators at the selected prompt condition."""
    result = evaluate.mcnemar_exact(
        primary_frame["pred"].tolist(), secondary_frame["pred"].tolist(), primary_frame["gold"].tolist()
    )
    return pd.DataFrame(
        [
            {
                "system_a": primary_frame["model_name"].iloc[0],
                "system_b": secondary_frame["model_name"].iloc[0],
                **result,
            }
        ]
    )


def select_model(model_mcnemar_df, hp=HYPERPARAMETERS):
    """Select the Arm 3 generator, returning (model_name, tie_break_fired).

    Mirrors the Arm 2 encoder rule: if McNemar does not resolve the pair at
    p < 0.05 the comparison is reported as unresolved and the clinical model is
    kept. This can retain the lower-scoring model.
    """
    row = model_mcnemar_df.iloc[0]
    primary = selected_model_name(hp)
    if row.p_value < 0.05:
        winner = row.system_a if row.accuracy_a > row.accuracy_b else row.system_b
        return winner, False
    return primary, True


def build_prompt_table(prompts_by_mode):
    """Concatenate prompt strings for persistence in `arm3_prompts.txt`."""
    sections = []
    for mode, prompts in prompts_by_mode.items():
        sections.append(f"### {mode}")
        for prompt in prompts:
            sections.append(flatten_messages(prompt) if isinstance(prompt, list) else prompt)
        sections.append("")
    return "\n".join(sections).strip() + "\n"


In [ ]:
%%writefile src/amlh/results.py
"""The frozen test run: scoring, cross-arm comparison and error analysis.

Everything here runs after every hyperparameter in `config.py` is frozen, and
nothing in this module selects anything. The test set is reached only through
`data.load_test`, which drops `answer`; `assert_no_answer_column` re-checks it
at each entry point.
"""

import pandas as pd
from sklearn.metrics import f1_score

from amlh import arm1_experiments as ae
from amlh import evaluate
from amlh.config import ARTEFACTS_DIR, HYPERPARAMETERS

# Ranking depth for the saved test predictions. Matches the validation side, so
# the test frames carry the same top_1..top_20 columns and Arm 3 has a shortlist.
SHORTLIST_DEPTH = 20

ARM_LABELS = {
    "arm1_tfidf_knn": "Arm 1 — TF-IDF + k-NN retrieval",
    "arm2_bio_clinicalbert": "Arm 2 — Bio_ClinicalBERT fine-tune",
    "arm3_llm_rerank": "Arm 3 — shortlist + MediPhi selection",
}

TEST_PREDICTION_FILES = {
    "arm1_tfidf_knn": "arm1_test_predictions.csv",
    "arm2_bio_clinicalbert": "arm2_test_predictions.csv",
    "arm3_llm_rerank": "arm3_test_predictions.csv",
}


def assert_no_answer_column(df, name="test"):
    """Raise if a frame still carries test-side answers.

    `data.load_test` already drops the column, so this should never fire; it
    catches any later code that reads the test CSV directly.
    """
    if "answer" in df.columns:
        raise ValueError(
            f"{name} frame carries an `answer` column. Test-side answers are never an "
            "inference-time input — load via data.load_test, which drops it."
        )


def build_test_predictions(fit_df, test_df, hp=HYPERPARAMETERS):
    """Arm 1's frozen top-1 and top-20 shortlist over the test questions.

    Reuses `arm1_experiments.build_val_predictions`, which works on any query
    frame, so the test and validation paths are the same code at the same
    hyperparameters. `fit_df` is `split_fit`, not the full training set, so the
    tested model is the model that was validated.
    """
    assert_no_answer_column(test_df)
    return ae.build_val_predictions(fit_df, test_df, hp, depth=SHORTLIST_DEPTH)


def assert_reproduces_arm1_test(shortlists, reference_path=None, tolerance=0):
    """Check a rebuilt Arm 1 test shortlist against the saved predictions.

    A degraded index still produces plausible-looking shortlists, so rank 1 is
    compared item for item before any GPU time is spent. TF-IDF and
    NearestNeighbors are deterministic on identical input, so `tolerance=0`.
    """
    reference_path = reference_path or ARTEFACTS_DIR / TEST_PREDICTION_FILES["arm1_tfidf_knn"]
    reference = pd.read_csv(reference_path)
    expected = reference["pred"].tolist()
    if len(shortlists) != len(expected):
        raise ValueError(f"rebuilt {len(shortlists)} shortlists, reference holds {len(expected)} rows")

    mismatches = [i for i, (s, e) in enumerate(zip(shortlists, expected)) if s[0] != e]
    if len(mismatches) > tolerance:
        raise ValueError(
            f"rebuilt Arm 1 test shortlist disagrees with {reference_path.name} on "
            f"{len(mismatches)}/{len(expected)} items (tolerance {tolerance}). The index is "
            "not the frozen one — check doc coverage and the QLAD variant before proceeding."
        )
    print(f"shortlist reproduces frozen Arm 1 test top-1 on {len(expected) - len(mismatches)}/{len(expected)} items")


def load_available_arms(artefacts_dir=ARTEFACTS_DIR):
    """Load whichever arms' test predictions exist, so this section still runs
    when an arm has not been produced yet."""
    frames = {}
    for arm, filename in TEST_PREDICTION_FILES.items():
        path = artefacts_dir / filename
        if path.exists():
            frames[arm] = pd.read_csv(path)
    return frames


def assert_query_aligned(frames):
    """Refuse to compare arms whose rows are not the same items in the same order.

    McNemar needs matched pairs, so gold labels are compared position by position
    rather than row order being trusted.
    """
    reference_name, reference = next(iter(frames.items()))
    gold = reference["gold"].tolist()
    for name, frame in frames.items():
        if len(frame) != len(gold):
            raise ValueError(f"{name} has {len(frame)} rows, {reference_name} has {len(gold)}")
        if frame["gold"].tolist() != gold:
            raise ValueError(f"{name} gold labels are not aligned with {reference_name}")
    return gold


def score_arms(frames, seed=None):
    """Test accuracy per arm with a bootstrap 95% CI, plus the ranked metrics.

    Accuracy is the headline metric the brief asks for; Top-5 and MRR are
    reported because the error analysis uses them.
    """
    from amlh.config import SEED

    seed = SEED if seed is None else seed
    gold = assert_query_aligned(frames)
    top_cols = [f"top_{i}" for i in range(1, SHORTLIST_DEPTH + 1)]

    rows = []
    for arm, frame in frames.items():
        pred = frame["pred"].tolist()
        ci = evaluate.bootstrap_accuracy_ci(pred, gold, seed=seed)
        row = {
            "arm": arm,
            "label": ARM_LABELS.get(arm, arm),
            "n": len(gold),
            "accuracy": sum(p == g for p, g in zip(pred, gold)) / len(gold),
            "ci_low": ci["ci_low"],
            "ci_high": ci["ci_high"],
            # macro-F1 needs only top-1, so every arm gets one
            "macro_f1": f1_score(gold, pred, average="macro", zero_division=0),
        }
        available = [c for c in top_cols if c in frame.columns]
        if available:
            ranked = [[lab for lab in r if isinstance(lab, str)] for r in frame[available].values.tolist()]
            scored = evaluate.score_ranked(ranked, gold)
            row.update({"acc_at_5": scored["acc_at_5"], "mrr": scored["mrr"]})
        else:
            # None, not a missing key: a blank cell means this arm produces no
            # ranking (Arm 3 returns one label), not that a value went astray.
            row.update({"acc_at_5": None, "mrr": None})
        rows.append(row)
    return pd.DataFrame(rows)


def pairwise_mcnemar(frames):
    """Every pairwise McNemar over the matched test items. All pairs are reported,
    since the brief asks for a comparison rather than a single chosen system."""
    from itertools import combinations

    gold = assert_query_aligned(frames)
    rows = []
    for a, b in combinations(frames, 2):
        result = evaluate.mcnemar_exact(frames[a]["pred"].tolist(), frames[b]["pred"].tolist(), gold)
        rows.append({"system_a": a, "system_b": b, **result})
    return pd.DataFrame(rows)


def validation_test_gap(test_scores, val_comparison_path=None):
    """Pair each arm's validation accuracy against its test accuracy.

    Measured per arm rather than assumed to be the same for all three. A
    diagnostic of the hold-out, computed after the test run; it selects nothing.
    """
    val_comparison_path = val_comparison_path or ARTEFACTS_DIR / "cross_arm_val_comparison.csv"
    val = pd.read_csv(val_comparison_path)[["arm", "accuracy"]].rename(columns={"accuracy": "val_accuracy"})
    merged = test_scores[["arm", "label", "accuracy"]].rename(columns={"accuracy": "test_accuracy"})
    merged = merged.merge(val, on="arm", how="left")
    merged["optimism"] = merged["val_accuracy"] - merged["test_accuracy"]
    return merged


def prefix_family_sizes(label_space):
    """Members per prefix family (`baby_`, `pregnancy_`, …) across the label space.

    Counted over all labels, not those present in one split: a model may predict
    any class, so whether a gold label has a sibling is a property of the label
    space rather than of the sample.
    """
    sizes = {}
    for label in label_space:
        prefix = label.split("_")[0]
        sizes[prefix] = sizes.get(prefix, 0) + 1
    return sizes


def family_error_summary(frame, label_space):
    """Within-family error rate, among errors where such a confusion was possible.

    A gold label whose prefix family has one member cannot be confused with a
    sibling, so it contributes a certain zero to the numerator while still
    counting in the denominator. Conditioning on the family being multi-member
    keeps the rate about the models rather than about the class composition of
    the split. `within_family_share` is None, not 0.0, when nothing was possible.
    """
    sizes = prefix_family_sizes(label_space)
    errors = frame[frame["pred"] != frame["gold"]]
    possible = [sizes.get(g.split("_")[0], 0) > 1 for g in errors["gold"]]
    within = [
        g.split("_")[0] == p.split("_")[0]
        for g, p, ok in zip(errors["gold"], errors["pred"], possible)
        if ok
    ]
    n_possible = sum(possible)
    return {
        "n_errors": len(errors),
        "n_family_error_possible": n_possible,
        "n_within_family": sum(within),
        "within_family_share": (sum(within) / n_possible) if n_possible else None,
    }


def confusion_pairs(frame, top_n=15, label_space=None):
    """Most frequent (gold, predicted) error pairs, with a same-family flag.

    An error inside `baby_*` is a different failure from one across unrelated
    conditions. Passing `label_space` adds `family_error_possible`, without which
    an all-False `same_family` column cannot be distinguished from gold labels
    that had no sibling. This frame is truncated to `top_n`, so use
    `family_error_summary` for rates.
    """
    errors = frame[frame["pred"] != frame["gold"]]
    counts = (
        errors.groupby(["gold", "pred"]).size().reset_index(name="n").sort_values("n", ascending=False)
    )
    counts["same_family"] = [
        g.split("_")[0] == p.split("_")[0] for g, p in zip(counts["gold"], counts["pred"])
    ]
    if label_space is not None:
        sizes = prefix_family_sizes(label_space)
        counts["family_error_possible"] = [sizes.get(g.split("_")[0], 0) > 1 for g in counts["gold"]]
    return counts.head(top_n).reset_index(drop=True)


def rerank_decomposition(arm3_frame):
    """Split Arm 3's items by what the LLM did with the shortlist it was given.

    - `kept_rank_1` — the LLM agreed with the retriever, so accuracy is Arm 1's.
    - `fell_back` — no shortlist label matched, so Arm 1's top-1 was returned.
    - `moved_off_rank_1` — the LLM overrode the retriever.

    `arm1_accuracy` per stratum is what the shortlist alone would have scored on
    the same items, so the gap on `moved_off_rank_1` is what the LLM cost or won.
    """
    frame = arm3_frame.copy()
    correct = frame["pred"] == frame["gold"]
    arm1_correct = frame["arm1_pred"] == frame["gold"]

    fell_back = frame["fallback_fired"].astype(bool)
    moved = (~fell_back) & (frame["parsed_rank"] > 1)
    kept = (~fell_back) & (frame["parsed_rank"] == 1)

    rows = []
    for name, mask in (("kept_rank_1", kept), ("fell_back", fell_back), ("moved_off_rank_1", moved)):
        n = int(mask.sum())
        rows.append(
            {
                "stratum": name,
                "n": n,
                "share": n / len(frame) if len(frame) else 0.0,
                "arm3_accuracy": float(correct[mask].mean()) if n else 0.0,
                "arm1_accuracy": float(arm1_correct[mask].mean()) if n else 0.0,
            }
        )
    out = pd.DataFrame(rows)
    out["accuracy_delta"] = out["arm3_accuracy"] - out["arm1_accuracy"]

    moved_n = int(moved.sum())
    gained = int((moved & correct & ~arm1_correct).sum())
    lost = int((moved & ~correct & arm1_correct).sum())
    print(
        f"interventions: {moved_n} | rescued {gained} | broke {lost} | "
        f"precision {gained / moved_n:.3f}" if moved_n else "no interventions"
    )
    return out


def worked_examples(frames, n=5, seed=None):
    """Sample correct and incorrect test predictions per arm, as the brief asks.
    Drawn with a fixed seed so the report's examples are stable across reruns."""
    from amlh.config import SEED

    seed = SEED if seed is None else seed
    rows = []
    for arm, frame in frames.items():
        correct = frame[frame["pred"] == frame["gold"]]
        wrong = frame[frame["pred"] != frame["gold"]]
        for outcome, subset in (("correct", correct), ("incorrect", wrong)):
            take = subset.sample(min(n, len(subset)), random_state=seed)
            for row in take.itertuples(index=False):
                rows.append(
                    {
                        "arm": arm,
                        "outcome": outcome,
                        "question": row.question,
                        "gold": row.gold,
                        "pred": row.pred,
                    }
                )
    return pd.DataFrame(rows)


In [ ]:
%%writefile src/amlh/eda.py
"""Diagnostics for the validation-test gap and the structure of the label space.

`sibling_homogeneity` and `novelty_calibrated_eval` read `test.disease`. They
describe how validation and test differ and select nothing.
"""

from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def label_family(diseases):
    """Shared-prefix family per disease label (text before the first `_`)."""
    return diseases.str.split("_").str[0]


def ambiguous_examples(train, n=4):
    """Up to `n` example question strings that map to more than one disease."""
    dup = train[train.duplicated("question", keep=False)]
    grouped = dup.groupby("question").disease.unique()
    examples = []
    for question, diseases in grouped.items():
        if len(diseases) <= 1:
            continue
        examples.append({"question": question, "diseases": sorted(diseases)})
        if len(examples) >= n:
            break
    return examples


def near_duplication_similarities(train, test):
    """Highest cosine similarity from each test question to any train question.
    Same recipe as the audit's near-duplication check, returned for plotting."""
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(train.question)
    sim = cosine_similarity(vec.transform(test.question), vec.transform(train.question))
    return sim.max(axis=1)


@dataclass
class SiblingHomogeneity:
    sib: list[float]  # train question -> nearest same-class sibling
    own: list[float]  # test question -> nearest own-class train question
    other: list[float]  # test question -> nearest other-class train question
    summary: pd.DataFrame


def sibling_homogeneity(train, test):
    """Phrasing similarity within a class, compared with train against test."""
    vec = TfidfVectorizer(sublinear_tf=True).fit(train.question)
    Xtr = vec.transform(train.question)
    Xte = vec.transform(test.question)

    sib = []
    for _, idx in train.groupby("disease").groups.items():
        idx = np.asarray(idx)
        if len(idx) < 2:
            continue
        S = cosine_similarity(Xtr[idx])
        np.fill_diagonal(S, -1)
        sib.extend(S.max(axis=1).tolist())

    by_disease = train.groupby("disease").groups
    own = [
        float(cosine_similarity(Xte[i], Xtr[np.asarray(by_disease[d])]).max())
        for i, d in enumerate(test.disease)
    ]

    S_all = cosine_similarity(Xte, Xtr)
    train_disease = train.disease.to_numpy()
    other = [
        float(S_all[i][train_disease != d].max()) for i, d in enumerate(test.disease)
    ]

    summary = pd.DataFrame(
        [
            {
                "comparison": "train question -> nearest same-class sibling",
                "mean_cosine": round(float(np.mean(sib)), 3),
            },
            {
                "comparison": "test question -> nearest own-class train question",
                "mean_cosine": round(float(np.mean(own)), 3),
            },
            {
                "comparison": "test question -> nearest other-class train question",
                "mean_cosine": round(float(np.mean(other)), 3),
            },
        ]
    )
    return SiblingHomogeneity(sib=sib, own=own, other=other, summary=summary)


def wrong_class_closer_fraction(own, other):
    """Fraction of test questions where the nearest other-class train question
    is more similar than the nearest own-class one."""
    return float(np.mean(np.asarray(other) > np.asarray(own)))


def novelty_calibrated_eval(fit, val, prune_threshold, k=20, vec_kwargs=None):
    """Prune fit questions that are near-siblings of each val question before a
    similarity-weighted k-NN vote, pushing validation novelty toward the test
    level. Returns (validation accuracy, median own-class novelty) at this
    threshold. Declared diagnostic use only — selects no hyperparameter.
    """
    vec = TfidfVectorizer(**(vec_kwargs or {"ngram_range": (1, 1), "sublinear_tf": True}))
    vec.fit(fit.question)
    Xf = vec.transform(fit.question)
    Xq = vec.transform(val.question)
    S = cosine_similarity(Xq, Xf)
    fit_disease = fit.disease.to_numpy()

    correct = []
    novelty = []
    for i, gold in enumerate(val.disease):
        s = S[i].copy()
        keep = s < prune_threshold
        s[~keep] = -1
        votes = defaultdict(float)
        for j in np.argsort(-s)[:k]:
            if s[j] > 0:
                votes[fit_disease[j]] += s[j]
        correct.append(max(votes, key=votes.get) == gold if votes else False)
        own_sims = S[i][(fit_disease == gold) & keep]
        novelty.append(float(own_sims.max()) if own_sims.size else 0.0)

    return round(float(np.mean(correct)), 3), round(float(np.median(novelty)), 3)


In [ ]:
%%writefile src/amlh/diagrams.py
"""Workflow diagrams for the report (Figures 4 and 5).

The brief asks for a diagram of each method's workflow. Drawing them in code
means every hyperparameter shown is read from `config.HYPERPARAMETERS`, so a
diagram cannot fall out of step with the frozen configuration. Boxes are placed
on a unit grid in axes coordinates.
"""

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from amlh.config import FIGURES_DIR, HYPERPARAMETERS

# muted colours, still distinguishable in greyscale if the report is printed
INPUT_FACE = "#e8eef5"
PROCESS_FACE = "#ffffff"
FROZEN_FACE = "#dce8dd"
OUTPUT_FACE = "#f3e6dd"
EDGE = "#33414f"
TEXT = "#16202b"
MUTED = "#5c6a78"
FLAG = "#a8422f"


def _box(ax, x, y, w, h, title, subtitle=None, face=PROCESS_FACE, fontsize=8.5):
    """One rounded node. `subtitle` holds the frozen hyperparameters, in smaller
    type so the diagram reads as a flow first."""
    ax.add_patch(
        FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.004,rounding_size=0.012",
            linewidth=0.9, edgecolor=EDGE, facecolor=face, zorder=2,
        )
    )
    cy = y + h / 2
    if subtitle:
        ax.text(x + w / 2, cy + h * 0.17, title, ha="center", va="center",
                fontsize=fontsize, color=TEXT, zorder=3)
        ax.text(x + w / 2, cy - h * 0.21, subtitle, ha="center", va="center",
                fontsize=fontsize - 1.7, color=MUTED, family="monospace", zorder=3)
    else:
        ax.text(x + w / 2, cy, title, ha="center", va="center",
                fontsize=fontsize, color=TEXT, zorder=3)


def _arrow(ax, start, end, linestyle="-", color=EDGE, rad=0.0):
    ax.add_patch(
        FancyArrowPatch(
            start, end, arrowstyle="-|>", mutation_scale=11,
            linewidth=0.9, color=color, linestyle=linestyle,
            connectionstyle=f"arc3,rad={rad}", zorder=1,
        )
    )


def _canvas(figsize):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    return fig, ax


def draw_arm1_workflow(hp=HYPERPARAMETERS, save_to=None):
    """F4 — Arm 1: TF-IDF + k-NN retrieval, and the shortlist it hands to Arm 3."""
    fig, ax = _canvas((9.2, 3.6))

    ax.text(0.0, 0.99, "Figure F4 — Arm 1: TF-IDF retrieval over a QLAD class-blob index",
            ha="left", va="top", fontsize=10, color=TEXT)
    ax.text(0.0, 0.925,
            "Green = frozen in config.py. Training answers and NHS documents are indexed as "
            "class evidence; at prediction time the only input is the question.",
            ha="left", va="top", fontsize=7.2, color=MUTED)

    _box(ax, 0.01, 0.56, 0.175, 0.22, "split_fit", "8,691 questions", face=INPUT_FACE)
    _box(ax, 0.01, 0.17, 0.175, 0.22, "NHS documents", "906 x .txt", face=INPUT_FACE)

    _box(ax, 0.225, 0.36, 0.19, 0.26,
         f"Index — {hp.index_variant}", f"{hp.index_scheme}\n906 class blobs", face=FROZEN_FACE)
    _box(ax, 0.455, 0.36, 0.185, 0.26,
         "TF-IDF", f"ngram {hp.ngram_range}\nmin_df={hp.min_df}", face=FROZEN_FACE)
    _box(ax, 0.68, 0.36, 0.175, 0.26,
         "Cosine k-NN", f"k={hp.k_neighbors}", face=FROZEN_FACE)

    _box(ax, 0.885, 0.56, 0.105, 0.22, "Top-1", "prediction", face=OUTPUT_FACE)
    _box(ax, 0.885, 0.17, 0.105, 0.22, f"Top-{hp.shortlist_k}", "to Arm 3", face=OUTPUT_FACE)

    _arrow(ax, (0.185, 0.67), (0.225, 0.55))
    _arrow(ax, (0.185, 0.28), (0.225, 0.43))
    _arrow(ax, (0.415, 0.49), (0.455, 0.49))
    _arrow(ax, (0.640, 0.49), (0.680, 0.49))
    _arrow(ax, (0.855, 0.54), (0.885, 0.66))
    _arrow(ax, (0.855, 0.44), (0.885, 0.32), linestyle=(0, (3, 2)))

    _box(ax, 0.36, 0.03, 0.375, 0.09,
         "Test question — the only inference-time input", face=INPUT_FACE, fontsize=8)
    _arrow(ax, (0.548, 0.12), (0.548, 0.36))

    fig.tight_layout()
    if save_to is not False:
        path = save_to or FIGURES_DIR / "fig4_arm1_workflow.png"
        fig.savefig(path, dpi=200, bbox_inches="tight")
        print(f"wrote {path}")
    return fig


def draw_arms23_workflow(hp=HYPERPARAMETERS, save_to=None):
    """F5 — Arms 2 and 3 together, so the shared Arm 1 dependency is visible.

    Arm 3 is not an independent system: it re-ranks Arm 1's shortlist and returns Arm 1's
    own top-1 whenever parsing finds no shortlist label. Drawing it apart from Arm 1 would
    misrepresent what it is, and would hide why its accuracy tracks Arm 1's.
    """
    fig, ax = _canvas((9.2, 4.7))

    ax.text(0.0, 0.99, "Figure F5 — Arm 2 (fine-tuned encoder) and Arm 3 (shortlist + LLM)",
            ha="left", va="top", fontsize=10, color=TEXT)

    # --- Arm 2 lane ---
    ax.text(0.0, 0.845, "ARM 2", ha="left", va="center", fontsize=8,
            color=MUTED, family="monospace")
    _box(ax, 0.075, 0.760, 0.155, 0.17, "split_fit", "8,691 questions", face=INPUT_FACE)
    _box(ax, 0.265, 0.760, 0.185, 0.17,
         "WordPiece tokeniser", f"max_length={hp.max_length}", face=FROZEN_FACE)
    _box(ax, 0.485, 0.760, 0.215, 0.17,
         "Bio_ClinicalBERT", f"lr={hp.learning_rate:g}  bs={hp.batch_size}\n906-way head",
         face=FROZEN_FACE)
    _box(ax, 0.735, 0.760, 0.185, 0.17,
         "Fine-tune", f"epoch {hp.num_epochs} selected", face=FROZEN_FACE)
    _arrow(ax, (0.230, 0.845), (0.265, 0.845))
    _arrow(ax, (0.450, 0.845), (0.485, 0.845))
    _arrow(ax, (0.700, 0.845), (0.735, 0.845))

    # --- shared input ---
    _box(ax, 0.365, 0.550, 0.27, 0.11, "Test question", face=INPUT_FACE, fontsize=8.5)
    _arrow(ax, (0.560, 0.660), (0.827, 0.760), rad=-0.15)

    # --- Arm 3 lane ---
    ax.text(0.0, 0.400, "ARM 3", ha="left", va="center", fontsize=8,
            color=MUTED, family="monospace")
    _box(ax, 0.075, 0.315, 0.165, 0.17,
         "Arm 1 shortlist", f"frozen, top-{hp.shortlist_k}", face=FROZEN_FACE)
    _box(ax, 0.275, 0.315, 0.185, 0.17,
         f"{hp.prompt_mode} prompt",
         f"T={hp.llm_temperature:g}  new={hp.arm3_max_new_tokens}", face=FROZEN_FACE)
    _box(ax, 0.495, 0.315, 0.175, 0.17, "MediPhi-Guidelines", "generate", face=FROZEN_FACE)
    _box(ax, 0.705, 0.315, 0.155, 0.17, "Parse to a\nshortlist label")

    _arrow(ax, (0.240, 0.400), (0.275, 0.400))
    _arrow(ax, (0.460, 0.400), (0.495, 0.400))
    _arrow(ax, (0.670, 0.400), (0.705, 0.400))
    _arrow(ax, (0.440, 0.550), (0.320, 0.485), rad=0.15)

    _box(ax, 0.885, 0.550, 0.105, 0.17, "Prediction", face=OUTPUT_FACE, fontsize=8.5)
    _arrow(ax, (0.920, 0.760), (0.920, 0.722))
    _arrow(ax, (0.860, 0.400), (0.920, 0.550), rad=-0.2)

    # The fallback: a parse failure returns Arm 1's top-1 unchanged.
    _arrow(ax, (0.782, 0.315), (0.158, 0.315), linestyle=(0, (3, 2)), color=FLAG, rad=-0.22)
    ax.text(0.470, 0.168, "no shortlist label matched: return Arm 1 top-1 unchanged",
            ha="center", va="center", fontsize=7.2, color=FLAG)

    ax.text(0.0, 0.065,
            "Green = frozen in config.py. Dashed red = the fallback path. Arm 3 is not "
            "independent of Arm 1 — it re-ranks Arm 1's shortlist and returns Arm 1's own\n"
            "top-1 whenever parsing finds no candidate, which is why its accuracy is anchored "
            "to Arm 1's.",
            ha="left", va="top", fontsize=7.2, color=MUTED)

    fig.tight_layout()
    if save_to is not False:
        path = save_to or FIGURES_DIR / "fig5_arms23_workflow.png"
        fig.savefig(path, dpi=200, bbox_inches="tight")
        print(f"wrote {path}")
    return fig


In [ ]:
import sys

sys.path.insert(0, "src")

from amlh.config import ARTEFACTS_DIR, FIGURES_DIR, HYPERPARAMETERS, PROJECT_ROOT, SEED, set_seed

# PROJECT_ROOT is derived from config.py's own location, so no path is hardcoded.
print(f"SEED         = {SEED}")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"data present = {(PROJECT_ROOT / 'data' / 'patient_qa_classification_train.csv').is_file()}")

PRECOMPUTED_DIR = ARTEFACTS_DIR / "precomputed"
mark("0. setup")

# §1 Dataset and preprocessing

*Backs report §2.1 (dataset), §2.2 (preprocessing, Table 1), §3.1 (EDA, Figures 1–3).*

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

from amlh import eda, features
from amlh.data import load_test, load_train, make_validation_split, run_integrity_audit

set_seed()
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 60)

## §1.1 Load and audit

`load_test` drops the `answer` column at the loader, so nothing downstream can read a
test-side answer. The audit reports class counts, unseen labels and train-test
near-duplication.


In [ ]:
train = load_train()
test = load_test()

print(f"train columns: {list(train.columns)}")
print(f"test  columns: {list(test.columns)}  <- no `answer`")

audit = run_integrity_audit(train, test)
print(json.dumps(audit, indent=2))

## §1.2 NHS document coverage and boilerplate

The `D` component of the Arm 1 index is the NHS reference document for each class. Six labels
break the naming convention (`Bronchitis`, `Pneumonia`, …), so filenames are resolved by
lowercasing. `features._strip_boilerplate` removes navigation text, image credits and review
dates — roughly 22% of each raw document — before anything is indexed.

In [ ]:
coverage = features.doc_coverage(train.disease.unique())
print(json.dumps(coverage, indent=2))
assert coverage["n_found"] == coverage["n_total"] == 906, "NHS document coverage is incomplete"

with open(ARTEFACTS_DIR / "doc_coverage.json", "w") as f:
    json.dump(coverage, f, indent=2)

# how much boilerplate stripping removes, over the whole corpus
raw_chars = clean_chars = 0
for disease in train.disease.unique():
    raw_chars += len(features.load_class_doc_raw(disease))
    clean_chars += len(features.load_class_doc(disease))

print(f"\nboilerplate stripping over all {train.disease.nunique()} documents: "
      f"{raw_chars:,} chars -> {clean_chars:,} ({1 - clean_chars / raw_chars:.1%} removed)")
print(f"\nexample -- gallstones, first 200 chars after cleaning:")
print(features.load_class_doc("gallstones")[:200] + "...")

## §1.3 Validation split

A single stratified hold-out of 200 items over 102 classes, matching the shape of the test
set. `make_validation_split` allocates the 200-item quota across classes up front, so every
selected class has at least one validation item while keeping at least two rows in the fit
half — no class can disappear from the index.

The split is shipped in the archive rather than regenerated. `make_validation_split` is
deterministic within an environment, but it does not reproduce the committed split on a
current numpy/pandas. The cell below shows both: the shipped split has exactly the structure
this algorithm produces, and a live call is deterministic in-process, but the draw differs.
Every result in the report was computed on the shipped split, so §2 onwards uses it.


In [ ]:
fit = pd.read_csv(ARTEFACTS_DIR / "split_fit.csv")
val = pd.read_csv(ARTEFACTS_DIR / "split_val.csv")
print(f"shipped split: fit={len(fit)} ({fit.disease.nunique()} classes) | "
      f"val={len(val)} ({val.disease.nunique()} classes) | test={len(test)}")

# does the shipped split have the structure this algorithm produces?
base, extra = divmod(200, 102)
per_class = val.disease.value_counts().value_counts().to_dict()
print(f"quota structure: base={base}, extra={extra} -> expect {extra} classes with {base + 1} "
      f"items and {102 - extra} with {base}")
print(f"shipped         : {per_class}")
assert per_class == {base + 1: extra, base: 102 - extra}, "shipped split is not this algorithm's output"

# a live call, compared against the shipped file
regenerated = make_validation_split(train, seed=SEED)
again = make_validation_split(train, seed=SEED)
overlap = len(set(regenerated.val.question) & set(val.question))
print(f"\nlive call is deterministic in-process : "
      f"{regenerated.val.question.tolist() == again.val.question.tolist()}")
print(f"live call vs shipped, items in common : {overlap}/{len(val)}")
print("=> the algorithm matches; the draw does not. §2 onwards uses the shipped split.")

## §1.4 Figure 1 — class support, question length, label families, near-duplication

*Report Figure 1.* Panel (a) is the fact that shapes the whole design: class support is
**near-uniform** at around 10 questions per class rather than long-tailed, so there is very
little per-class data for a classifier to learn a boundary from.

Panels (b) and (d) read test questions. This is a distributional diagnostic and selects
nothing.


In [ ]:
for d in (train, test):
    d["q_len"] = d.question.str.split().str.len()

cnt = train.disease.value_counts()
fam = eda.label_family(pd.Series(sorted(train.disease.unique())))
fam_counts = fam.value_counts()
near_dup_sim = eda.near_duplication_similarities(train, test)

print(f"class support: min {cnt.min()} | median {cnt.median():.0f} | max {cnt.max()} | sd {cnt.std():.2f}")

fig, ax = plt.subplots(2, 2, figsize=(13, 9))

sns.histplot(cnt.values, bins=range(int(cnt.min()), int(cnt.max()) + 2), ax=ax[0, 0])
ax[0, 0].set(xlabel="Training questions per class", ylabel="Number of classes",
             title=f"(a) Near-uniform class support ({len(cnt)} classes)")

sns.histplot(train.q_len, bins=30, stat="density", ax=ax[0, 1], label="train")
sns.histplot(test.q_len, bins=30, stat="density", ax=ax[0, 1], label="test", color="darkorange")
ax[0, 1].legend()
ax[0, 1].set(xlabel="Question length (words)", title="(b) Question length, train vs test")

fam_counts.head(12).plot.barh(ax=ax[1, 0])
ax[1, 0].invert_yaxis()
ax[1, 0].set(xlabel="Labels in family", title="(c) Top-12 label families")

ax[1, 1].hist(near_dup_sim, bins=30)
ax[1, 1].axvline(0.9, ls="--", c="r")
ax[1, 1].set(xlabel="Max cosine to any training question",
             title="(d) Train-test near-duplication")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig1_eda.png", dpi=150)
plt.show()

## §1.5 Preprocessing evidence — Table 1

Question-length percentiles fix BERT's `max_length=48` (§2.2): they are measured here, not
chosen. The label-family and ambiguity tables quantify how fine-grained the label space is —
355 of the 906 labels share a prefix family, and identical question strings map to more than
one disease, which is an irreducible error floor.

In [ ]:
q_len_percentiles = train.q_len.describe(percentiles=[.5, .9, .95, .99])
print(q_len_percentiles)
q_len_percentiles.to_csv(ARTEFACTS_DIR / "question_length_percentiles.csv", header=["value"])

family_sizes = fam_counts.rename_axis("family").reset_index(name="n_labels")
family_sizes.to_csv(ARTEFACTS_DIR / "label_family_sizes.csv", index=False)
print()
print(f"labels in a multi-member family: {family_sizes.loc[family_sizes.n_labels > 1, 'n_labels'].sum()}"
      f" of {len(fam)}")
print(family_sizes.head(6).to_string(index=False))

In [ ]:
examples = eda.ambiguous_examples(train, n=4)
print("identical question strings mapping to more than one disease:")
for ex in examples:
    print(f"  '{ex['question']}' -> {ex['diseases']}")

pd.DataFrame(examples).to_csv(ARTEFACTS_DIR / "ambiguous_examples.csv", index=False)

## §1.6 Why the hold-out over-estimates — sibling homogeneity

*Report §3.1, Figure 2.* This measurement explains the validation-to-test gap in §5. The
~10 questions per disease were generated in one pass, so a held-out question repeats the
phrasing of the ones left in training while a test question does not. Cross-validation
would not fix it, since those siblings sit inside every fold.

These cells read `test.disease` as a diagnostic. They select nothing.


In [ ]:
homogeneity = eda.sibling_homogeneity(train, test)
print(homogeneity.summary)

wrong_closer = eda.wrong_class_closer_fraction(homogeneity.own, homogeneity.other)
print(f"\nwrong class closer for {wrong_closer:.1%} of test questions")

homogeneity.summary.to_csv(ARTEFACTS_DIR / "sibling_homogeneity.csv", index=False)

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(homogeneity.sib, label="train q -> same-class sibling", fill=True)
sns.kdeplot(homogeneity.own, label="test q -> own-class train q", fill=True)
sns.kdeplot(homogeneity.other, label="test q -> other-class train q", fill=True)
plt.xlabel("Cosine similarity")
plt.title("Sibling phrasing homogeneity explains the val-test gap")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig2_novelty.png", dpi=150)
plt.show()

In [ ]:
# Pruning near-sibling fit questions from the index until validation novelty matches the
# test level gives a more realistic accuracy estimate. A secondary diagnostic.
test_novelty = round(float(np.median(homogeneity.own)), 3)
print(f"target novelty (test) = {test_novelty}")

rows = []
for t in (1.01, 0.70, 0.60, 0.50, 0.45, 0.40):
    val_accuracy, median_novelty = eda.novelty_calibrated_eval(fit, val, prune_threshold=t)
    rows.append({"prune_threshold": t, "val_accuracy": val_accuracy,
                 "median_own_class_novelty": median_novelty})

calibration = pd.DataFrame(rows)
print(calibration.to_string(index=False))
calibration.to_csv(ARTEFACTS_DIR / "novelty_calibration.csv", index=False)

## §1.7 Persist the enriched audit

The split files are the shipped ones and are not rewritten here. Only the audit is written,
now carrying the coverage and homogeneity measurements.


In [ ]:
audit["doc_coverage"] = coverage
audit["sibling_homogeneity"] = homogeneity.summary.set_index("comparison").mean_cosine.to_dict()
audit["wrong_class_closer_fraction"] = round(wrong_closer, 4)
audit["test_median_own_class_novelty"] = test_novelty

with open(ARTEFACTS_DIR / "integrity_audit.json", "w") as f:
    json.dump(audit, f, indent=2)
print("wrote artefacts/integrity_audit.json")
print("split_fit.csv and split_val.csv left as shipped -- see §1.3")

## §1.8 Figure 3 — PCA of TF-IDF question vectors

*Report Figure 3.* The ten largest classes projected to two dimensions. Classes overlap heavily,
which is the visual form of the same problem the sibling measurement quantifies.

In [ ]:
big_classes = cnt.head(10).index
subset = train[train.disease.isin(big_classes)]

vec = TfidfVectorizer(sublinear_tf=True, min_df=2).fit(subset.question)
pcs = PCA(n_components=2, random_state=SEED).fit_transform(vec.transform(subset.question).toarray())

plt.figure(figsize=(8, 6))
sns.scatterplot(x=pcs[:, 0], y=pcs[:, 1], hue=subset.disease.values, s=25)
plt.title("PCA of TF-IDF question vectors, 10 largest classes")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig3_pca.png", dpi=150)
plt.show()

mark("1. data and preprocessing")

# §2 Arm 1 — TF-IDF retrieval + k-NN

*Backs report §2.3 (method), §4.2 (design choices), Figure 7.*

Patient questions are represented as TF-IDF vectors and classified by nearest-neighbour
retrieval against an index built from the training half. Everything in this section is chosen on
**validation only**; no test quantity is read anywhere in §2.

Selection uses a **within-1-SE, prefer-simplest** rule rather than raw argmax throughout. With
n = 200 validation items, SE ≈ 3.5pp, so a raw argmax would routinely pick a configuration the
hold-out cannot actually distinguish from a simpler one.

In [ ]:
from amlh import arm1_experiments as ae
from amlh import arm1_tfidf, evaluate
from amlh.data import make_hard_validation_split, make_random_split

set_seed()

fit = pd.read_csv(ARTEFACTS_DIR / "split_fit.csv")
val = pd.read_csv(ARTEFACTS_DIR / "split_val.csv")
print(f"fit={len(fit)} ({fit.disease.nunique()} classes) | val={len(val)} ({val.disease.nunique()} classes)")

## §2.1 Vectoriser grid

`ngram_range × sublinear_tf × min_df × stop_words × k` — 80 configurations, evaluated on the
question-only (`Q`) class-blob index. Both the argmax row and the within-1-SE selection are
printed, so the effect of the rule is visible rather than asserted.

In [ ]:
grid_Q = ae.run_vectoriser_grid(
    fit, val, "Q",
    ngram_ranges=[(1, 1), (1, 2)],
    sublinear_tf_opts=[False, True],
    min_dfs=[1, 2],
    stop_words_opts=[None, "english"],
    ks=[1, 3, 5, 10, 20],
)
grid_Q.to_csv(ARTEFACTS_DIR / "arm1_grid_Q.csv", index=False)
print(f"{len(grid_Q)} configs run")
grid_Q.sort_values("accuracy", ascending=False).head(10)

In [ ]:
argmax_row_Q = grid_Q.loc[grid_Q["accuracy"].idxmax()]
selected_row_Q = ae.select_within_one_se(grid_Q, n_val=len(val))
print("argmax row:")
print(argmax_row_Q)
print()
print("selected (simplest within 1 SE) row:")
print(selected_row_Q)

best_vec_kwargs = {
    "ngram_range": selected_row_Q["ngram_range"],
    "sublinear_tf": selected_row_Q["sublinear_tf"],
    "min_df": selected_row_Q["min_df"],
    "stop_words": selected_row_Q["stop_words"],
}
best_k = int(selected_row_Q["k"])
print()
print("best_vec_kwargs (Q, from step 1):", best_vec_kwargs)
print("best_k (Q, from step 1):", best_k)

## §2.2 Preprocessing ablation — lemmatisation and stop words

*Report §2.2, Table 1.* `raw` versus spaCy lemmatisation with and without stop-word removal, at
the selected vectoriser config. The same within-1-SE rule applies: lemmatisation is only adopted
if it beats `raw` by more than the hold-out can resolve.

In [ ]:
preprocessing_ablation = ae.run_preprocessing_ablation(fit, val, best_vec_kwargs, best_k)
preprocessing_ablation.to_csv(ARTEFACTS_DIR / "arm1_preprocessing_ablation.csv", index=False)
print(preprocessing_ablation.to_string(index=False))

In [ ]:
best_acc_ablation = preprocessing_ablation["accuracy"].max()
se_ablation = (best_acc_ablation * (1 - best_acc_ablation) / len(val)) ** 0.5
raw_acc = preprocessing_ablation.loc[preprocessing_ablation["preprocessing"] == "raw", "accuracy"].item()

if raw_acc >= best_acc_ablation - se_ablation:
    best_preprocessing = "raw"
else:
    best_preprocessing = preprocessing_ablation.loc[
        preprocessing_ablation["accuracy"].idxmax(), "preprocessing"
    ]

print(f"best accuracy: {best_acc_ablation:.3f} (SE={se_ablation:.3f}) | raw accuracy: {raw_acc:.3f}")
print("selected preprocessing (prefer raw within 1 SE):", best_preprocessing)

## §2.3 Index variants — Q / QL / QLA / QLAD

*Report §2.3, §4.2.* The largest single lever in Arm 1. Each class contributes one indexed
"blob"; the variant controls what goes into it:

| | Component |
|---|---|
| `Q` | the class's training **q**uestions |
| `L` | the class **l**abel text |
| `A` | the training-side **a**nswers (permitted — these are training data) |
| `D` | the NHS reference **d**ocument |

`QLAD` needs full document coverage, checked before use.

In [ ]:
coverage = features.doc_coverage(fit["disease"].unique())
assert coverage["n_found"] == coverage["n_total"] == 906, "expected full 906/906 NHS doc coverage"
print(f"NHS document coverage: {coverage['n_found']}/{coverage['n_total']}")

index_variants = ae.run_index_variant_comparison(
    fit, val, ["Q", "QL", "QLA", "QLAD"], best_vec_kwargs, best_k
)
index_variants.to_csv(ARTEFACTS_DIR / "arm1_index_variants.csv", index=False)
print(index_variants.to_string(index=False))

### §2.3a A second hold-out for tie-breaks

The standard hold-out comes from the same one-pass generation as the fit split, so it
inherits the sibling phrasing measured in §1.6. The **hard** hold-out is drawn only from
training rows whose question shares no substantive word with its own disease label, which
is closer to the test set. It uses training data alone; no test text or label is read.

Its role is narrow and was fixed before these numbers were seen. The standard hold-out
decides wherever it separates the candidates. Only where they fall within 1 SE of each
other does the hard hold-out break the tie.


In [ ]:
hard_split = make_hard_validation_split(load_train(), seed=SEED)
print(f"hard fit={len(hard_split.fit)} | hard val={len(hard_split.val)} "
      f"({hard_split.val.disease.nunique()} classes)")

index_variants_hard = ae.run_index_variant_comparison(
    hard_split.fit, hard_split.val, ["Q", "QL", "QLA", "QLAD"], best_vec_kwargs, best_k
)
index_variants_hard.to_csv(ARTEFACTS_DIR / "arm1_index_variants_hardval.csv", index=False)
print(index_variants_hard.to_string(index=False))

In [ ]:
std_best = index_variants["accuracy"].max()
std_se = (std_best * (1 - std_best) / len(val)) ** 0.5
std_tied = index_variants.loc[index_variants["accuracy"] >= std_best - std_se, "variant"].tolist()

if len(std_tied) == 1:
    best_variant = std_tied[0]
    selection_basis = "standard hold-out (discriminates without a tie-break)"
else:
    contenders = index_variants_hard[index_variants_hard["variant"].isin(std_tied)]
    best_variant = contenders.loc[contenders["accuracy"].idxmax(), "variant"]
    hard_se = (contenders["accuracy"].max() * (1 - contenders["accuracy"].max()) / len(hard_split.val)) ** 0.5
    runner_up = contenders["accuracy"].nlargest(2).iloc[-1]
    selection_basis = (
        f"hard hold-out tie-break over {std_tied}; margin over runner-up "
        f"{contenders['accuracy'].max() - runner_up:.3f} vs SE {hard_se:.3f}"
    )

print(f"standard-val accuracies: {dict(zip(index_variants['variant'], index_variants['accuracy']))}")
print(f"within-1-SE set at {std_best:.3f} +/- {std_se:.3f}: {std_tied}")
print(f"hard-val accuracies:     {dict(zip(index_variants_hard['variant'], index_variants_hard['accuracy']))}")
print(f"selected index variant:  {best_variant}  [{selection_basis}]")

### §2.3b Variant-specific vectoriser re-check

Blob length differs enormously by variant — short questions versus long NHS prose — so the
optimum tuned on `Q` need not transfer. The grid is re-run restricted to the selected variant and
re-selected within 1 SE. If the winner differs, the variant-specific one is used downstream.

In [ ]:
grid_variant = ae.run_vectoriser_grid(
    fit, val, best_variant,
    ngram_ranges=[(1, 1), (1, 2)],
    sublinear_tf_opts=[False, True],
    min_dfs=[1, 2],
    stop_words_opts=[None, "english"],
    ks=[1, 3, 5, 10, 20],
)
grid_variant.to_csv(ARTEFACTS_DIR / f"arm1_grid_{best_variant}.csv", index=False)
selected_row_variant = ae.select_within_one_se(grid_variant, n_val=len(val))
print(selected_row_variant)

In [ ]:
variant_vec_kwargs = {
    "ngram_range": selected_row_variant["ngram_range"],
    "sublinear_tf": selected_row_variant["sublinear_tf"],
    "min_df": selected_row_variant["min_df"],
    "stop_words": selected_row_variant["stop_words"],
}
variant_k = int(selected_row_variant["k"])

config_unchanged = variant_vec_kwargs == best_vec_kwargs and variant_k == best_k
print("variant-specific config matches the step-1 config:", config_unchanged)

if not config_unchanged:
    print("Using the variant-specific winner downstream.")
    best_vec_kwargs = variant_vec_kwargs
    best_k = variant_k

print("final best_vec_kwargs:", best_vec_kwargs)
print("final best_k:", best_k)

## §2.4 Indexing scheme — class blob vs additive per row

Does spreading the index over one row per training example, rather than one blob per class,
change accuracy? Reported on both hold-outs. Here the standard hold-out separates the two
schemes by more than its own SE, so the tie-break does not apply.


In [ ]:
indexing_scheme = ae.run_indexing_scheme_comparison(fit, val, best_variant, best_vec_kwargs, best_k)
indexing_scheme.to_csv(ARTEFACTS_DIR / "arm1_indexing_scheme.csv", index=False)

indexing_scheme_hard = ae.run_indexing_scheme_comparison(
    hard_split.fit, hard_split.val, best_variant, best_vec_kwargs, best_k
)
indexing_scheme_hard.to_csv(ARTEFACTS_DIR / "arm1_indexing_scheme_hardval.csv", index=False)

print("standard hold-out:")
print(indexing_scheme.to_string(index=False))
print("\nhard hold-out:")
print(indexing_scheme_hard.to_string(index=False))

In [ ]:
std_scheme_best = indexing_scheme["accuracy"].max()
std_scheme_se = (std_scheme_best * (1 - std_scheme_best) / len(val)) ** 0.5
std_scheme_margin = std_scheme_best - indexing_scheme["accuracy"].nlargest(2).iloc[-1]

hard_scheme_best = indexing_scheme_hard["accuracy"].max()
hard_scheme_se = (hard_scheme_best * (1 - hard_scheme_best) / len(hard_split.val)) ** 0.5
hard_scheme_margin = hard_scheme_best - indexing_scheme_hard["accuracy"].nlargest(2).iloc[-1]

std_pick = indexing_scheme.loc[indexing_scheme["accuracy"].idxmax(), "scheme"]
hard_pick = indexing_scheme_hard.loc[indexing_scheme_hard["accuracy"].idxmax(), "scheme"]

print(f"standard-val: {std_pick} by {std_scheme_margin:.3f} (SE {std_scheme_se:.3f}) "
      f"-> {'decisive' if std_scheme_margin > std_scheme_se else 'not decisive'}")
print(f"hard-val:     {hard_pick} by {hard_scheme_margin:.3f} (SE {hard_scheme_se:.3f}) "
      f"-> {'decisive' if hard_scheme_margin > hard_scheme_se else 'not decisive'}")

if hard_pick != std_pick and hard_scheme_margin > hard_scheme_se and std_scheme_margin <= std_scheme_se:
    best_scheme = hard_pick
    scheme_basis = "hard hold-out (standard hold-out did not discriminate)"
else:
    best_scheme = std_pick
    scheme_basis = "standard hold-out (hard hold-out does not overturn it at >1 SE)"

print(f"selected indexing scheme: {best_scheme}  [{scheme_basis}]")

## §2.5 Split-robustness check

The top-3 configurations re-run under an unstratified split across three seeds. **Absolute
accuracy is not comparable across split designs** — the fit/validation composition differs — so
the only thing under test is whether the *ranking* survives.

In [ ]:
top3_configs = (
    grid_Q.sort_values("accuracy", ascending=False)
    .head(3)[["ngram_range", "sublinear_tf", "min_df", "stop_words", "k"]]
    .to_dict("records")
)
for i, cfg in enumerate(top3_configs, start=1):
    print(i, cfg)

split_robustness = ae.run_split_robustness(load_train(), top3_configs, variant="Q", seeds=(42, 43, 44))
split_robustness.to_csv(ARTEFACTS_DIR / "arm1_split_robustness.csv", index=False)

mean_by_rank = split_robustness.groupby("config_rank")["accuracy"].mean()
print("\nmean unstratified-split accuracy across 3 seeds, by config_rank:")
print(mean_by_rank.to_string())
ranking_preserved = list(mean_by_rank.sort_values(ascending=False).index) == list(mean_by_rank.index)
print(f"\nconfig ranking order preserved under the random split: {ranking_preserved}")

## §2.6 Supervised baselines

*Report §2.3.* Per-row TF-IDF classification with LinearSVC, LogisticRegression and
RandomForest, at the selected vectoriser config. These contrast learned decision boundaries
against neighbour matching under ~10 examples per class. They are comparison points, not
candidates for the frozen system.

In [ ]:
supervised_baselines = ae.run_supervised_baselines(fit, val, best_vec_kwargs)
supervised_baselines.to_csv(ARTEFACTS_DIR / "arm1_supervised_baselines.csv", index=False)
print(supervised_baselines.to_string(index=False))

## §2.7 Figure 7 — accuracy–coverage curve

*Report Figure 7, §4.3.* As the similarity threshold for abstention rises, coverage falls and the
accuracy of the predictions that are still made rises. This is the quantitative basis for
discussing abstention as a safety mechanism in a clinical setting.

In [ ]:
if best_scheme == "class_blob":
    index_texts, index_labels = features.build_index(fit, best_variant)
else:
    index_texts, index_labels = features.build_index_additive(fit, best_variant)

ranked, top_sim = arm1_tfidf.knn_rank(
    index_texts, index_labels, val["question"].tolist(), best_k, best_vec_kwargs
)
gold = val["disease"].tolist()

thresholds = [round(0.05 * i, 2) for i in range(21)]
coverage_curve = evaluate.accuracy_coverage_curve(ranked, gold, top_sim, thresholds)
coverage_curve.to_csv(ARTEFACTS_DIR / "arm1_coverage_curve.csv", index=False)

fig, ax1 = plt.subplots(figsize=(6, 4))
ax1.plot(coverage_curve["threshold"], coverage_curve["accuracy"], marker="o",
         color="tab:blue", label="accuracy")
ax1.set_xlabel("similarity threshold")
ax1.set_ylabel("accuracy (retained predictions)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(coverage_curve["threshold"], coverage_curve["coverage"], marker="s",
         color="tab:orange", label="coverage")
ax2.set_ylabel("coverage (fraction retained)", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

fig.suptitle("Arm 1 accuracy-coverage curve (validation)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig7_coverage_curve.png", dpi=150)
plt.show()

## §2.8 Freeze Arm 1

The values selected above, assembled from the DataFrames rather than typed in. These are the
values already recorded in `config.py`'s `HYPERPARAMETERS`; the assertion confirms this run
reproduces them, so everything downstream reads the frozen config rather than this section's
local variables.

In [ ]:
frozen_arm1 = {
    "ngram_range": tuple(int(x) for x in best_vec_kwargs["ngram_range"]),
    "min_df": int(best_vec_kwargs["min_df"]),
    "max_df": 1.0,
    "sublinear_tf": bool(best_vec_kwargs["sublinear_tf"]),
    "stop_words": best_vec_kwargs["stop_words"],
    "lemmatise": best_preprocessing != "raw",
    "index_variant": str(best_variant),
    "index_scheme": str(best_scheme),
    "k_neighbors": int(best_k),
}
print("selected by this run:")
for key, value in frozen_arm1.items():
    recorded = getattr(HYPERPARAMETERS, key)
    flag = "ok " if recorded == value else "DIFFERS"
    print(f"  {flag} {key:16s} = {value!r}   (config.py: {recorded!r})")

mismatched = [k for k, v in frozen_arm1.items() if getattr(HYPERPARAMETERS, k) != v]
assert not mismatched, f"this run did not reproduce the frozen Arm 1 config: {mismatched}"
print("\nall Arm 1 hyperparameters reproduce config.py")

## §2.9 Per-item predictions and the shortlist ceiling

At the frozen `k_neighbors=1` the similarity-weighted vote holds one label, so Arm 1's ranked
list has length 1 and its Top-5 and MRR columns repeat accuracy. They are ranking metrics for
Arm 2 but not here.

`knn_rank`'s `depth` parameter appends lower-confidence labels below the unchanged head, which
gives Arm 3 a 20-label shortlist. The assertion confirms the head is unchanged on all 200
items.

In [ ]:
SHORTLIST_DEPTH = 20

ranked_frozen, top_sim_frozen = ae.frozen_ranking(fit, val, HYPERPARAMETERS)
ranked_deep, top_sim_deep = ae.frozen_ranking(fit, val, HYPERPARAMETERS, depth=SHORTLIST_DEPTH)

assert [r[0] for r in ranked_frozen] == [r[0] for r in ranked_deep], \
    "ranking-depth path changed a frozen top-1 prediction"
assert top_sim_frozen == top_sim_deep, "ranking-depth path changed the reported top similarity"
print(f"depth={SHORTLIST_DEPTH} leaves all {len(val)} frozen top-1 predictions unchanged")

arm1_val_predictions = ae.build_val_predictions(fit, val, HYPERPARAMETERS, depth=SHORTLIST_DEPTH)
arm1_val_predictions.to_csv(ARTEFACTS_DIR / "arm1_val_predictions.csv", index=False)

arm1_val_accuracy = (arm1_val_predictions["pred"] == arm1_val_predictions["gold"]).mean()
print(f"Arm 1 validation accuracy: {arm1_val_accuracy:.4f}")

arm1_shortlist_ceiling = ae.shortlist_ceiling(arm1_val_predictions, [1, 5, 10, 20])
arm1_shortlist_ceiling.to_csv(ARTEFACTS_DIR / "arm1_shortlist_ceiling.csv", index=False)
print()
print(arm1_shortlist_ceiling.to_string(index=False))

mark("2. Arm 1")

# §3 Arm 2 — fine-tuned Bio_ClinicalBERT

*Backs report §2.4 (method), Figure 6, §4.2.*

`question` → WordPiece (`max_length=48`) → encoder → 906-way linear classification head →
argmax. `answer` is never an input to this arm. Labels are encoded against the **full 906-class
training universe**, not the classes present in the fit split, so the output layer can predict
anything the frozen test run might need.

Requires a GPU. Validation only — no test quantity is read in §3.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from amlh import arm2_bert as ab

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU'}")
assert device.type == "cuda", "Arms 2 and 3 need a GPU -- Runtime > Change runtime type > T4"

## §3.1 Label encoding and tokenisation

`max_length=48` comes from §1.5's question-length percentiles (mean 8.4 words, 99th percentile
18) — fixed from the data, not tuned. The truncation rate it produces is printed here and belongs
in report §2.2.

In [ ]:
set_seed()

label_to_id, id_to_label = ab.encode_labels(load_train())
assert len(label_to_id) == 906
print(f"{len(label_to_id)} classes encoded over the full training universe")

MAX_LENGTH = HYPERPARAMETERS.max_length
LEARNING_RATE = HYPERPARAMETERS.learning_rate
BATCH_SIZE = HYPERPARAMETERS.batch_size
SWEEP_EPOCHS = 24
MODEL_NAMES = ["emilyalsentzer/Bio_ClinicalBERT", "bert-base-uncased"]

sanity_tokeniser = AutoTokenizer.from_pretrained("bert-base-uncased")
rate = ab.truncation_rate(fit["question"].tolist(), sanity_tokeniser, MAX_LENGTH)
print(f"truncation rate at max_length={MAX_LENGTH}: {rate:.4f}")
print(f"lr={LEARNING_RATE} | batch_size={BATCH_SIZE}  (standard fine-tuning defaults, held fixed "
      f"across both encoders and not swept -- at SE ~3.5pp a 200-item hold-out cannot resolve a "
      f"learning-rate grid)")

## §3.2 Encoder comparison sweep — *gated*

`run_model_ablation` trains Bio_ClinicalBERT and `bert-base-uncased` for 24 epochs with
**identical** hyperparameters, seed and epoch budget, so the only thing that varies is the
checkpoint. That is what makes the in-domain-pretraining claim measurable rather than asserted.

The recorded run took 3,998 s (66.6 min) on a T4 across both encoders, as logged in
`arm2_model_ablation.csv`. With `RUN_FULL_SELECTION = False` the sweep is loaded from
`artefacts/precomputed/`; the selection rules in §3.3 run live on it either way.

In [ ]:
if RUN_FULL_SELECTION:
    def report(model_name, row):
        print(f"{model_name.split('/')[-1]} | epoch {row['epoch'] + 1}/{SWEEP_EPOCHS} | "
              f"train_loss {row['train_loss']:.4f} | val_loss {row['val_loss']:.4f} | "
              f"val_acc {row['val_accuracy']:.4f}")

    set_seed()
    start = time.perf_counter()
    ablation_summary, histories, ranked_by_epoch = ab.run_model_ablation(
        fit, val, label_to_id, MODEL_NAMES,
        lr=LEARNING_RATE, batch_size=BATCH_SIZE, epochs=SWEEP_EPOCHS,
        max_length=MAX_LENGTH, seed=SEED, on_epoch_end=report,
    )
    print(f"\ntotal wall clock for both encoders: {time.perf_counter() - start:.1f}s")

    histories["emilyalsentzer/Bio_ClinicalBERT"].to_csv(
        ARTEFACTS_DIR / "arm2_history_bioclinicalbert.csv", index=False)
    histories["bert-base-uncased"].to_csv(
        ARTEFACTS_DIR / "arm2_history_bertbase.csv", index=False)
    ablation_summary.to_csv(ARTEFACTS_DIR / "arm2_model_ablation.csv", index=False)

    encoder_predictions = None  # rebuilt from ranked_by_epoch in §3.3
else:
    print("RUN_FULL_SELECTION = False -- loading the recorded sweep from artefacts/precomputed/")
    histories = {
        "emilyalsentzer/Bio_ClinicalBERT": pd.read_csv(PRECOMPUTED_DIR / "arm2_history_bioclinicalbert.csv"),
        "bert-base-uncased": pd.read_csv(PRECOMPUTED_DIR / "arm2_history_bertbase.csv"),
    }
    ablation_summary = pd.read_csv(PRECOMPUTED_DIR / "arm2_model_ablation.csv")
    ranked_by_epoch = None

print()
print(ablation_summary.to_string(index=False))

### §3.2a Figure 6 — training and validation loss

*Report Figure 6.* Both curves for both encoders. Report §4.2 reads the train/validation gap as
evidence of memorisation under ~10 examples per class, which is only legible with both plotted.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, history in histories.items():
    short = name.split("/")[-1]
    ax.plot(history["epoch"], history["train_loss"], marker="o", label=f"{short} train")
    ax.plot(history["epoch"], history["val_loss"], marker="o", linestyle="--", label=f"{short} val")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("Arm 2 - training/validation loss")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig6_loss_curves.png", dpi=150)
plt.show()

## §3.3 Epoch and encoder selection — always live

Two decisions, both made on the standard hold-out. Arm 2 does not use §2.3a's hard
hold-out; that applies to Arm 1's index selection only.

1. **Epoch** — among epochs within 1 SE of the best validation accuracy, take the earliest.
   This buys the peak's accuracy for less training.
2. **Encoder** — McNemar's exact test over the discordant pairs, rather than an accuracy
   difference judged against a single-proportion SE. Both encoders answer the same 200
   items, so the comparison is paired and the items they agree on carry no evidence.

If McNemar returns p ≥ 0.05 the comparison is reported as unresolved and Bio_ClinicalBERT is
kept, on the stated preference for an in-domain clinical encoder on a clinical task. This can
retain the lower-scoring encoder, and here it does. Unlike §2.3a's rule, this one was written
down after the comparison was seen, which the report notes.


In [ ]:
selected_by_encoder = {
    name: ab.select_best_epoch_within_one_se(history, n_val=len(val))
    for name, history in histories.items()
}
for name, row in selected_by_encoder.items():
    peak = histories[name]["val_accuracy"].max()
    se = (peak * (1 - peak) / len(val)) ** 0.5
    n_within = int((histories[name]["val_accuracy"] >= peak - se).sum())
    print(f"{name}")
    print(f"  peak val_accuracy {peak:.4f} (SE {se:.4f}) -> {n_within} of {len(histories[name])} "
          f"epochs within 1 SE")
    print(f"  selected epoch {int(row['epoch'])}, val_accuracy {row['val_accuracy']:.4f}")

In [ ]:
SHORT_NAME = {"emilyalsentzer/Bio_ClinicalBERT": "bioclinicalbert", "bert-base-uncased": "bertbase"}

if RUN_FULL_SELECTION:
    # taken from the rankings train_model kept during the sweep, so no retraining is needed
    encoder_predictions = {}
    for model_name, row in selected_by_encoder.items():
        ranked_at_epoch = ranked_by_epoch[model_name][int(row["epoch"])]
        frame = pd.DataFrame({"question": val["question"], "gold": val["disease"]})
        frame["pred"] = [r[0] for r in ranked_at_epoch]
        for i in range(5):
            frame[f"top_{i + 1}"] = [r[i] if i < len(r) else None for r in ranked_at_epoch]
        frame.to_csv(ARTEFACTS_DIR / f"arm2_val_predictions_{SHORT_NAME[model_name]}.csv", index=False)
        encoder_predictions[model_name] = frame
else:
    encoder_predictions = {
        name: pd.read_csv(PRECOMPUTED_DIR / f"arm2_val_predictions_{short}.csv")
        for name, short in SHORT_NAME.items()
    }

for name, frame in encoder_predictions.items():
    print(f"{SHORT_NAME[name]:16s} accuracy {(frame['pred'] == frame['gold']).mean():.4f}")

encoder_mcnemar = evaluate.mcnemar_exact(
    encoder_predictions["emilyalsentzer/Bio_ClinicalBERT"]["pred"].tolist(),
    encoder_predictions["bert-base-uncased"]["pred"].tolist(),
    val["disease"].tolist(),
)
print("\nMcNemar exact -- a = Bio_ClinicalBERT, b = bert-base-uncased, same 200 items:")
for key, value in encoder_mcnemar.items():
    print(f"  {key}: {value}")

pd.DataFrame([{"system_a": "emilyalsentzer/Bio_ClinicalBERT",
               "system_b": "bert-base-uncased", **encoder_mcnemar}]).to_csv(
    ARTEFACTS_DIR / "arm2_encoder_mcnemar.csv", index=False)

In [ ]:
ALPHA = 0.05

if encoder_mcnemar["p_value"] >= ALPHA:
    print(f"McNemar p = {encoder_mcnemar['p_value']:.4f} >= alpha = {ALPHA}")
    print("UNRESOLVED -- the hold-out does not separate these encoders. No winner is declared.")
    print("Tie-break: keep Bio_ClinicalBERT, the in-domain clinical encoder.")
    selected_model_name = "emilyalsentzer/Bio_ClinicalBERT"
    encoder_tie_break_fired = True
else:
    print(f"McNemar p = {encoder_mcnemar['p_value']:.4f} < alpha = {ALPHA}: the difference is reliable.")
    print("Tie-break does not fire; measured accuracy decides.")
    accs = {n: (f["pred"] == f["gold"]).mean() for n, f in encoder_predictions.items()}
    selected_model_name = max(accs, key=accs.get)
    encoder_tie_break_fired = False

selected_row = selected_by_encoder[selected_model_name]
selected_epochs = int(selected_row["epoch"]) + 1

print(f"\nselected encoder : {selected_model_name}")
print(f"selected epochs  : {selected_epochs} (0-indexed epoch {int(selected_row['epoch'])})")
if encoder_tie_break_fired:
    kept = (encoder_predictions[selected_model_name]["pred"]
            == encoder_predictions[selected_model_name]["gold"]).mean()
    other = [n for n in encoder_predictions if n != selected_model_name][0]
    other_acc = (encoder_predictions[other]["pred"] == encoder_predictions[other]["gold"]).mean()
    if other_acc > kept:
        print(f"NOTE: this retains the LOWER-scoring encoder ({kept:.3f} vs {other_acc:.3f} for "
              f"{other}). The tie-break selected it, not the accuracy.")

assert selected_model_name == HYPERPARAMETERS.bert_model_name, "encoder differs from config.py"
assert selected_epochs == HYPERPARAMETERS.num_epochs, "epoch count differs from config.py"
print("\nboth reproduce config.py")

## §3.4 Train the frozen model

Trained at the selected epoch count on `split_fit` only — **no refit on fit + val**, so the model
evaluated on test in §5 is the exact model that was validated here, and the validation→test
comparison in §5.6 describes one object rather than two.

The state dict is kept and the GPU copy released, so the encoder is not resident while §4 loads a
3.8B-parameter generator.

In [ ]:
set_seed()
start = time.perf_counter()

state_dict, frozen_history, _ = ab.train_model(
    fit, val, label_to_id, HYPERPARAMETERS.bert_model_name,
    lr=LEARNING_RATE, batch_size=BATCH_SIZE, epochs=selected_epochs,
    max_length=MAX_LENGTH, seed=SEED,
)
print(f"trained {selected_epochs} epochs in {(time.perf_counter() - start) / 60:.1f} min")
print(frozen_history.tail().to_string(index=False))

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    HYPERPARAMETERS.bert_model_name, num_labels=len(label_to_id)
)
model.load_state_dict(state_dict)
model = model.to(device)
tokeniser = AutoTokenizer.from_pretrained(HYPERPARAMETERS.bert_model_name)

# top_k=None ranks all 906 labels, so Arm 2's MRR is comparable with Arm 1's.
val_ranked = ab.predict_ranked(
    model, tokeniser, val["question"].tolist(), id_to_label,
    max_length=MAX_LENGTH, batch_size=BATCH_SIZE, device=device, top_k=None,
)
arm2_scores = evaluate.score_ranked(val_ranked, val["disease"].tolist())
print(arm2_scores)

# The refit replays the same seed and data, so its top-1 should reproduce the sweep's.
# Printed rather than asserted, since cuDNN kernel selection is not bit-deterministic.
captured_top1 = encoder_predictions[HYPERPARAMETERS.bert_model_name]["pred"].tolist()
agreement = sum(a == b[0] for a, b in zip(captured_top1, val_ranked)) / len(val)
print(f"refit vs sweep-captured top-1 agreement: {agreement:.4f}")

arm2_val_predictions = pd.DataFrame({"question": val["question"], "gold": val["disease"]})
arm2_val_predictions["pred"] = [r[0] for r in val_ranked]
for i in range(5):
    arm2_val_predictions[f"top_{i + 1}"] = [r[i] if i < len(r) else None for r in val_ranked]
arm2_val_predictions.to_csv(ARTEFACTS_DIR / "arm2_val_predictions.csv", index=False)

pd.DataFrame([{**arm2_scores, "model_name": HYPERPARAMETERS.bert_model_name,
               "epoch": selected_epochs - 1, "refit_agreement": agreement}]).to_csv(
    ARTEFACTS_DIR / "arm2_val_metrics.csv", index=False)

del model
torch.cuda.empty_cache()
print(f"encoder released | GPU allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
mark("3. Arm 2")

# §4 Arm 3 — retrieval shortlist + LLM re-ranking

*Backs report §2.4 (method), Table 2 (prompts), §3.3, §4.1.*

The frozen Arm 1 retriever hands the LLM a 20-label shortlist; the LLM picks one by name. This is
the Week 10 practical's diagnosis-selection task with the candidate list narrowed from all 906
diseases to the retriever's shortlist, and the generator is called through the same
`pipe(prompt, max_new_tokens=...)` interface.

The retriever's own top-1 is already a prediction, so **Arm 1 is the baseline this arm has to
beat**, not a component whose contribution can be assumed positive.

In [ ]:
from amlh import arm3_llm as a3
from amlh import results

set_seed()

shortlist_k = a3.shortlist_k(HYPERPARAMETERS)
n_shots = a3.n_shots(HYPERPARAMETERS)
primary_model = a3.selected_model_name(HYPERPARAMETERS)
secondary_model = a3.secondary_model_name(HYPERPARAMETERS)
prompt_modes = ["zero_shot", "few_shot", "cot"]

print({
    "shortlist_k": shortlist_k,
    "llm_temperature": a3.llm_temperature(HYPERPARAMETERS),
    "n_shots": n_shots,
    "max_new_tokens": a3.max_new_tokens_for("zero_shot"),
    "cot_max_new_tokens": a3.max_new_tokens_for("cot"),
    "primary_model": primary_model,
    "secondary_model": secondary_model,
})

## §4.1 The shortlist

`build_shortlist_ranking` runs the frozen Arm 1 configuration from `config.py`. The assertion
then checks rank 1 against §2.9's `arm1_val_predictions.csv`, item for item.

Rank 1 identifies the index that produced it, so a single disagreement stops the notebook
before any prompt is sent. This catches a shortlist built from anything other than the frozen
QLAD index — for example on a runtime where the NHS documents are missing, which would leave
`load_class_doc` returning empty strings and the index running as QLA.


In [ ]:
set_seed()
shortlist_rankings, top_sim = a3.build_shortlist_ranking(fit, val, HYPERPARAMETERS, depth=shortlist_k)

check = a3.assert_reproduces_arm1(shortlist_rankings, arm1_val_predictions)
print(f"shortlist top-1 reproduces Arm 1 on all {check['n']} items")

gold = val["disease"].tolist()
arm1_top1 = [r[0] for r in shortlist_rankings]
in_shortlist = sum(g in r for g, r in zip(gold, shortlist_rankings)) / len(gold)
print(f"Arm 1 top-1 accuracy on these items : {sum(p == g for p, g in zip(arm1_top1, gold)) / len(gold):.3f}")
print(f"gold present in the shortlist       : {in_shortlist:.3f}  (the ceiling the LLM selects against)")

## §4.2 Prompt budget — Table 2

*Report Table 2.* Three prompt conditions — `zero_shot`, `few_shot` (2 exemplars drawn from the
fit split only), and `cot`. Lengths are measured with the primary model's own tokeniser before any
condition runs, and the truncation rate at 512 tokens is printed rather than assumed.

`cot` gets 128 new tokens against 20 for the others, since chain-of-thought has to produce
reasoning as well as a final answer.

In [ ]:
set_seed()
examples = a3.build_examples(fit, n=n_shots, seed=SEED)
budget_tokeniser = AutoTokenizer.from_pretrained(primary_model)

prompts_by_mode = {}
budget_rows = []
for mode in prompt_modes:
    prompts = a3.build_prompts_for_condition(
        val, shortlist_rankings, mode, examples=examples if mode == "few_shot" else None
    )
    prompts_by_mode[mode] = prompts
    summary = a3.prompt_token_lengths(prompts, budget_tokeniser, max_length=512)
    budget_rows.append({
        "condition": mode,
        "min_tokens": summary["min"], "median_tokens": summary["median"],
        "mean_tokens": summary["mean"], "p95_tokens": summary["p95"],
        "max_tokens": summary["max"], "truncation_rate_512": summary["truncation_rate"],
    })

budget_df = pd.DataFrame(budget_rows)
budget_df.to_csv(ARTEFACTS_DIR / "arm3_prompt_budget.csv", index=False)
print(budget_df.to_string(index=False))

with open(ARTEFACTS_DIR / "arm3_prompts.txt", "w", encoding="utf-8") as f:
    f.write(a3.build_prompt_table(prompts_by_mode))

print()
print("--- one zero-shot prompt, as the model receives it ---")
print(a3.flatten_messages(prompts_by_mode["zero_shot"][0]))

## §4.3 Condition × model grid — *gated*

Three prompt conditions on two generators. **The recorded run took 1,448 s (24.1 min) on a T4**,
summed over the six cells from `arm3_prompt_conditions.csv`; `cot` dominates that at 758 s + 395 s
because of its 128-token budget. With `RUN_FULL_SELECTION = False` the six cells are loaded from
`artefacts/precomputed/`, and the selection rules in §4.4 run live on them either way.

In [ ]:
if RUN_FULL_SELECTION:
    def run_grid(model_name):
        frames, rows = {}, []
        _, model_obj, pipe = a3.load_generator(model_name, device=device)
        print(f"loaded {model_name} on {device}")
        for mode in prompt_modes:
            start = time.perf_counter()
            pred_df, metrics, _ = a3.run_condition(
                fit, val, HYPERPARAMETERS, mode=mode, pipe=pipe,
                examples=examples if mode == "few_shot" else None,
                shortlist_depth=shortlist_k, model_name=model_name,
                shortlist_rankings=shortlist_rankings, top_sim=top_sim,
            )
            metrics["wall_clock_sec"] = time.perf_counter() - start
            frames[mode] = pred_df
            rows.append(metrics)
            print(f"{mode:>10}: acc={metrics['accuracy']:.3f}  arm1={metrics['arm1_accuracy']:.3f}  "
                  f"fallback={metrics['fallback_rate']:.3f}  ({metrics['wall_clock_sec']:.0f}s)")
        # free the GPU before the next generator is loaded
        del model_obj, pipe
        torch.cuda.empty_cache()
        return frames, rows

    set_seed()
    condition_frames, primary_rows = run_grid(primary_model)
    secondary_frames, secondary_rows = run_grid(secondary_model)

    condition_df = pd.DataFrame(primary_rows)
    ablation_df = pd.concat([condition_df, pd.DataFrame(secondary_rows)], ignore_index=True)
    pd.concat(list(condition_frames.values()) + list(secondary_frames.values()),
              ignore_index=True).to_csv(ARTEFACTS_DIR / "arm3_val_predictions.csv", index=False)
else:
    print("RUN_FULL_SELECTION = False -- loading the recorded grid from artefacts/precomputed/")
    all_preds = pd.read_csv(PRECOMPUTED_DIR / "arm3_val_predictions.csv")
    ablation_df = pd.read_csv(PRECOMPUTED_DIR / "arm3_prompt_conditions.csv")

    def frames_for(model_name):
        return {
            mode: all_preds[(all_preds["condition"] == mode)
                            & (all_preds["model_name"] == model_name)].reset_index(drop=True)
            for mode in prompt_modes
        }

    condition_frames = frames_for(primary_model)
    secondary_frames = frames_for(secondary_model)
    condition_df = ablation_df[ablation_df["model_name"] == primary_model].reset_index(drop=True)

ablation_df.to_csv(ARTEFACTS_DIR / "arm3_prompt_conditions.csv", index=False)
print()
print(ablation_df.to_string(index=False))

### §4.3a Did the LLM earn its place?

`accuracy_minus_arm1` is the quantity that matters: the LLM re-ranks a shortlist whose top-1 is
already a prediction. McNemar pairs each condition against that top-1 over the same items.

In [ ]:
vs_arm1_df = a3.condition_vs_arm1_mcnemar(condition_frames)
vs_arm1_df.to_csv(ARTEFACTS_DIR / "arm3_vs_arm1_mcnemar.csv", index=False)
print(vs_arm1_df.to_string(index=False))

## §4.4 Two-stage selection — always live

A 6-cell grid on a 200-item hold-out (SE ≈ 3.5pp) cannot support six-way selection, so the
choice is made in two stages, in an order fixed before the run:

1. **Prompt condition**, on the primary generator alone, by pairwise McNemar. If no condition
   separates at p < 0.05 the comparison is reported as unresolved and `zero_shot` is kept as
   the simplest prompt — fewest tokens, no exemplar selection, lowest inference cost. This can
   retain a lower-scoring condition.
2. **Model**, at the selected condition, by McNemar over the same items. If p ≥ 0.05 the
   in-domain clinical model is kept, as in §3.3.

Unlike §3.3's encoder rule, this order was fixed before any condition was run. The remaining
grid cells are reported for completeness and select nothing.


In [ ]:
condition_mcnemar_df = a3.pairwise_condition_mcnemar(condition_frames)
selected_mode, condition_tie_break_fired = a3.select_prompt_mode(condition_df, condition_mcnemar_df)
condition_mcnemar_df.to_csv(ARTEFACTS_DIR / "arm3_condition_mcnemar.csv", index=False)

print("stage 1 -- prompt condition, on the primary generator:")
print(condition_mcnemar_df.to_string(index=False))
print()
print({"selected_mode": selected_mode, "tie_break_fired": condition_tie_break_fired})
if condition_tie_break_fired:
    smallest = condition_mcnemar_df.loc[condition_mcnemar_df["p_value"].idxmin()]
    print(f"UNRESOLVED: no condition separates at p < 0.05 (smallest p = {smallest['p_value']:.4f}, "
          f"{smallest['condition_a']} vs {smallest['condition_b']}).")
    print("Tie-break: keep zero_shot, the simplest prompt.")
    best_acc_mode = condition_df.loc[condition_df["accuracy"].idxmax()]
    if best_acc_mode["condition"] != selected_mode:
        chosen_acc = condition_df.loc[condition_df["condition"] == selected_mode, "accuracy"].item()
        print(f"NOTE: this retains the LOWER-scoring condition ({selected_mode} {chosen_acc:.3f} vs "
              f"{best_acc_mode['condition']} {best_acc_mode['accuracy']:.3f}). The tie-break selected it.")

In [ ]:
model_mcnemar_df = a3.model_mcnemar(condition_frames[selected_mode], secondary_frames[selected_mode])
selected_generator, model_tie_break_fired = a3.select_model(model_mcnemar_df, HYPERPARAMETERS)
model_mcnemar_df.to_csv(ARTEFACTS_DIR / "arm3_model_mcnemar.csv", index=False)

print(f"stage 2 -- model, at the selected condition ({selected_mode}):")
print(model_mcnemar_df.to_string(index=False))
print()
print({"selected_generator": selected_generator, "tie_break_fired": model_tie_break_fired})
if model_tie_break_fired:
    print("UNRESOLVED: McNemar does not separate the generators; the in-domain clinical "
          "model is kept.")
else:
    print("RESOLVED on measured accuracy -- the tie-break was not needed.")

assert selected_mode == HYPERPARAMETERS.prompt_mode, "prompt_mode differs from config.py"
assert selected_generator == HYPERPARAMETERS.arm3_model_name, "arm3_model_name differs from config.py"
print("\nboth reproduce config.py")

## §4.5 Run the frozen condition live

The selected cell — `zero_shot` on the frozen generator — re-run on validation now, so the
selected system is executed in this notebook rather than only loaded. The generator stays resident
for the test run in §5.5.

In [ ]:
set_seed()
# The generator stays loaded across cells, through the test run in §5.4, where it is freed.
_, arm3_model_obj, arm3_pipe = a3.load_generator(HYPERPARAMETERS.arm3_model_name, device=device)
print(f"loaded {HYPERPARAMETERS.arm3_model_name} on {device}")

start = time.perf_counter()
arm3_val_frozen, arm3_val_metrics, _ = a3.run_condition(
    fit, val, HYPERPARAMETERS,
    mode=HYPERPARAMETERS.prompt_mode, pipe=arm3_pipe, examples=examples,
    shortlist_depth=shortlist_k, model_name=HYPERPARAMETERS.arm3_model_name,
    shortlist_rankings=shortlist_rankings, top_sim=top_sim,
)
print(f"ran 200 validation items in {time.perf_counter() - start:.1f}s")
for key, value in arm3_val_metrics.items():
    print(f"  {key}: {value}")

recorded_acc = ablation_df.loc[
    (ablation_df["condition"] == HYPERPARAMETERS.prompt_mode)
    & (ablation_df["model_name"] == HYPERPARAMETERS.arm3_model_name), "accuracy"].item()
print(f"\nlive run {arm3_val_metrics['accuracy']:.4f} | recorded {recorded_acc:.4f} | "
      f"delta {abs(arm3_val_metrics['accuracy'] - recorded_acc):.4f}")

### §4.5a What the LLM did to the shortlist

*Report §3.3, §4.1.* Arm 3's headline accuracy blends three groups of items that behave
differently, so it is reported split. `arm1_accuracy` on each stratum is what the shortlist
alone would have scored on those same items.

The fallback rate is not a parser fault: none of the fallback outputs match any of the 906
labels, so they are free text the model invented. It is reported as a finding rather than
tuned away.


In [ ]:
arm3_val_decomposition = results.rerank_decomposition(condition_frames[selected_mode])
display(arm3_val_decomposition)

fallbacks = condition_frames[selected_mode][condition_frames[selected_mode]["fallback_fired"]]
label_universe = set(load_train()["disease"].unique())
matched = sum(str(r).strip() in label_universe for r in fallbacks["raw_output"])
print(f"\nfallback outputs matching any of the 906 labels: {matched} of {len(fallbacks)} "
      f"-- the parser is correct, these are hallucinations")

## §4.6 Cross-arm comparison on validation

*Report §3.2.* All three arms have now answered the same 200 validation items, so they can be
compared pairwise. This is the comparison §5.6 measures the optimism of each arm against, and it
is computed here, before any test question is read.

In [ ]:
from itertools import combinations

val_arms = {
    "arm1_tfidf_knn": arm1_val_predictions,
    "arm2_bio_clinicalbert": arm2_val_predictions,
    "arm3_llm_rerank": condition_frames[selected_mode],
}
val_gold = results.assert_query_aligned(val_arms)
print(f"query-aligned on {len(val_gold)} validation items across {len(val_arms)} arms\n")

rows = []
for name, frame in val_arms.items():
    pred = frame["pred"].tolist()
    ci = evaluate.bootstrap_accuracy_ci(pred, val_gold, seed=SEED)
    rows.append({"arm": name, "n": len(val_gold),
                 "accuracy": sum(p == g for p, g in zip(pred, val_gold)) / len(val_gold),
                 "ci_low": ci["ci_low"], "ci_high": ci["ci_high"]})
cross_arm_val = pd.DataFrame(rows)
cross_arm_val.to_csv(ARTEFACTS_DIR / "cross_arm_val_comparison.csv", index=False)
print(cross_arm_val.to_string(index=False))

mcnemar_rows = [
    {"system_a": a, "system_b": b,
     **evaluate.mcnemar_exact(val_arms[a]["pred"].tolist(), val_arms[b]["pred"].tolist(), val_gold)}
    for a, b in combinations(val_arms, 2)
]
cross_arm_val_mcnemar = pd.DataFrame(mcnemar_rows)
cross_arm_val_mcnemar.to_csv(ARTEFACTS_DIR / "cross_arm_val_mcnemar.csv", index=False)
print()
for row in cross_arm_val_mcnemar.itertuples(index=False):
    verdict = "RESOLVED" if row.p_value < 0.05 else "UNRESOLVED"
    print(f"{row.system_a:24s} vs {row.system_b:24s} p = {row.p_value:.4g}  -> {verdict}")

mark("4. Arm 3")

# §5 The frozen test run

*Backs report §3.2 (performance comparison), §3.3 (error analysis), Figures 8–9.*

This is the only section that evaluates on the test set, and it runs after every
hyperparameter is frozen. Nothing here selects anything: no variant, checkpoint, prompt or
threshold. Every choice was made on validation in §2–§4 and is read from
`config.HYPERPARAMETERS`.

Three conditions govern it:

1. **`answer` is never an input at prediction time.** The column is absent from the shipped
   CSV and `results.assert_no_answer_column` re-checks at each entry point.
2. **Models are fit on `split_fit` only**, with no refit on fit + val, so the tested model is
   the model that was validated and §5.6 describes one object. The data forgone is 200 of
   8,891 items (2.2%), small against a 3.5pp SE.
3. **No single system is nominated.** The brief asks for a comparison of algorithms, so all
   three arms are reported side by side.


## §5.1 Freeze check

Confirm every hyperparameter this run depends on is actually frozen. A `None` here would mean a
choice was never made and the test run would be silently inventing one.

Test *questions and labels* have already been read once, in §1.4–§1.6, for the distributional
diagnostics declared there; those select nothing. This is the first point at which a test question
is scored, and it comes after the freeze.

In [ ]:
from amlh import results  # repeated from §4 so §5 runs standalone

set_seed()

FROZEN_FIELDS = [
    "ngram_range", "min_df", "max_df", "sublinear_tf", "lemmatise",
    "index_variant", "index_scheme", "k_neighbors",
    "bert_model_name", "max_length", "learning_rate", "batch_size", "num_epochs",
    "shortlist_k", "llm_temperature", "prompt_mode", "n_shots",
    "arm3_model_name", "arm3_max_new_tokens",
]
frozen = {name: getattr(HYPERPARAMETERS, name) for name in FROZEN_FIELDS}
unset = [name for name, value in frozen.items() if value is None]
assert not unset, f"hyperparameters still unfrozen: {unset}"

print(f"SEED = {SEED}")
for name, value in frozen.items():
    print(f"  {name:24s} = {value!r}")
print(f"\nall {len(frozen)} hyperparameters frozen -- the first test question is SCORED from the "
      f"next cell on (§1.4-§1.6 read test for the declared diagnostics only)")

In [ ]:
test = load_test()
results.assert_no_answer_column(test)

print(f"fit  = {len(fit):5d} rows, {fit.disease.nunique()} classes")
print(f"test = {len(test):5d} rows, {test.disease.nunique()} classes")
print(f"test columns: {list(test.columns)}  <- no `answer`")
print(f"test classes absent from split_fit: {len(set(test.disease) - set(fit.disease))}")

## §5.2 Arm 1 on test

`require_doc_coverage` raises rather than degrading when the `D` component is missing. The
ranking uses the same `frozen_ranking` code path as the validation grid, at the same
hyperparameters, so the test and validation numbers describe one system.


In [ ]:
print(f"doc coverage: {features.require_doc_coverage(fit.disease.unique())}")

set_seed()
arm1_test = results.build_test_predictions(fit, test, HYPERPARAMETERS)
arm1_test.to_csv(ARTEFACTS_DIR / "arm1_test_predictions.csv", index=False)

print(f"Arm 1 test accuracy: {(arm1_test['pred'] == arm1_test['gold']).mean():.4f} "
      f"over {len(arm1_test)} items")
arm1_test[["question", "gold", "pred", "top_sim"]].head()

## §5.3 Arm 2 on test

The encoder is rebuilt from §3.4's state dict — the same weights, no retraining — then released
again so the generator has the GPU to itself.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    HYPERPARAMETERS.bert_model_name, num_labels=len(label_to_id)
)
model.load_state_dict(state_dict)
model = model.to(device)

truncation = ab.truncation_rate(test["question"].tolist(), tokeniser, HYPERPARAMETERS.max_length)
print(f"test truncation rate at max_length={HYPERPARAMETERS.max_length}: {truncation:.4f}")

test_ranked = ab.predict_ranked(
    model, tokeniser, test["question"].tolist(), id_to_label,
    max_length=HYPERPARAMETERS.max_length, batch_size=HYPERPARAMETERS.batch_size,
    device=device, top_k=None,
)

arm2_test = pd.DataFrame({"question": test["question"], "gold": test["disease"]})
arm2_test["pred"] = [r[0] for r in test_ranked]
for i in range(results.SHORTLIST_DEPTH):
    arm2_test[f"top_{i + 1}"] = [r[i] if i < len(r) else None for r in test_ranked]
arm2_test.to_csv(ARTEFACTS_DIR / "arm2_test_predictions.csv", index=False)

print(f"Arm 2 test accuracy: {(arm2_test['pred'] == arm2_test['gold']).mean():.4f}")

del model
torch.cuda.empty_cache()

## §5.4 Arm 3 on test

The shortlist is rebuilt on this runtime and checked against §5.2's
`arm1_test_predictions.csv` item for item before any prompt is sent, as in §4.1.


In [ ]:
set_seed()
features.require_doc_coverage(fit.disease.unique())

test_shortlists, test_top_sim = a3.build_shortlist_ranking(
    fit, test, HYPERPARAMETERS, depth=shortlist_k
)
results.assert_reproduces_arm1_test(test_shortlists)

gold_in_shortlist = sum(g in s for g, s in zip(test["disease"], test_shortlists)) / len(test)
print(f"shortlist depth {len(test_shortlists[0])} | gold present in {gold_in_shortlist:.3f} "
      f"of shortlists (Arm 3's ceiling)")

In [ ]:
set_seed()
start = time.perf_counter()

arm3_test, arm3_test_metrics, _ = a3.run_condition(
    fit, test, HYPERPARAMETERS,
    mode=HYPERPARAMETERS.prompt_mode, pipe=arm3_pipe, examples=examples,
    model_name=HYPERPARAMETERS.arm3_model_name,
    shortlist_rankings=test_shortlists, top_sim=test_top_sim,
)
arm3_test.to_csv(ARTEFACTS_DIR / "arm3_test_predictions.csv", index=False)

print(f"ran in {time.perf_counter() - start:.1f}s")
for key, value in arm3_test_metrics.items():
    print(f"  {key}: {value}")

# release the generator held open since §4.5
del arm3_model_obj, arm3_pipe
torch.cuda.empty_cache()
print(f"\ngenerator released | GPU allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## §5.5 Headline test results

*Report §3.2, Figure 9.* Accuracy is the headline metric the brief specifies, with a percentile
bootstrap 95% CI over resampled items. Top-5, macro-F1 and MRR ride along for the error analysis —
they explain *why* accuracy is capped and are not promoted to headline status.

In [ ]:
frames = results.load_available_arms()
missing = [arm for arm in results.TEST_PREDICTION_FILES if arm not in frames]
print("available:", list(frames))
if missing:
    print(f"MISSING: {missing} -- sections below report only the arms present.")

test_gold = results.assert_query_aligned(frames)
print(f"query-aligned on {len(test_gold)} test items")

test_scores = results.score_arms(frames)
test_scores.to_csv(ARTEFACTS_DIR / "arm_test_comparison.csv", index=False)
test_scores

## §5.6 Pairwise McNemar, and validation → test optimism

Every arm answers the same 200 questions, so the comparisons are paired and the items two arms
agree on carry no evidence about which is better. That is why a difference of two accuracies is
never judged against a single-proportion SE here.

The optimism table pairs each arm's validation accuracy (§4.6) against its test accuracy,
measured per arm rather than assumed to be the same for all three. It is a diagnostic of the
hold-out, computed after the test run, and selects nothing.

In [ ]:
test_mcnemar = results.pairwise_mcnemar(frames)
test_mcnemar.to_csv(ARTEFACTS_DIR / "arm_test_mcnemar.csv", index=False)

for row in test_mcnemar.itertuples(index=False):
    verdict = "RESOLVED" if row.p_value < 0.05 else "UNRESOLVED"
    print(f"{row.system_a:24s} vs {row.system_b:24s} p = {row.p_value:.4g}  -> {verdict}")
test_mcnemar

In [ ]:
gap = results.validation_test_gap(test_scores)
gap.to_csv(ARTEFACTS_DIR / "validation_test_gap.csv", index=False)
gap

## §5.7 Error analysis

*Report §3.3.*

### §5.7a Most frequent confusions

`same_family` flags whether gold and prediction share a label prefix; `family_error_possible`
flags whether that row's gold label had **any** sibling in the 906-label space to be confused
with. Both are needed — a `same_family` column of all-False means nothing on its own, because it
cannot be told apart from a set of gold labels that had no siblings in the first place.

In [ ]:
label_space = pd.read_csv(PROJECT_ROOT / "data" / "patient_qa_classification_train.csv")["disease"].unique()

for arm, frame in frames.items():
    pairs = results.confusion_pairs(frame, top_n=10, label_space=label_space)
    print()
    print(f"=== {results.ARM_LABELS.get(arm, arm)} - top confusions ===")
    print(pairs.to_string(index=False))
    pairs.to_csv(ARTEFACTS_DIR / f"{arm}_test_confusions.csv", index=False)

### §5.7b Family-internal error share — conditioned on being possible

Is the residual error *fine-grained* (confusing two conditions in the same family) or *coarse*
(missing the topic entirely)? An unconditioned share cannot answer it: a gold label whose prefix
family has only one member **cannot** be confused with a sibling, so it contributes a guaranteed
zero to the numerator while still inflating the denominator.

On validation 88 of 200 items have a gold label in a multi-member family; on test only 18 do.
`family_error_summary` reports that denominator alongside the count, and returns `None` rather
than `0.0` when nothing was possible. Read `n_family_error_possible` before the share: on test
it is too small to support a rate, which is why the report quotes this on validation only.

In [ ]:
family_errors = pd.DataFrame(
    [{"arm": arm, **results.family_error_summary(frame, label_space)} for arm, frame in frames.items()]
)
family_errors.to_csv(ARTEFACTS_DIR / "test_family_error_share.csv", index=False)

val_family_errors = pd.DataFrame(
    [{"arm": arm, **results.family_error_summary(frame, label_space)} for arm, frame in val_arms.items()]
)
val_family_errors.to_csv(ARTEFACTS_DIR / "val_family_error_share.csv", index=False)

print("TEST - denominator too small to support a rate:")
print(family_errors.to_string(index=False))
print()
print("VALIDATION - denominator supports a rate:")
print(val_family_errors.to_string(index=False))

### §5.7c What the LLM did to the shortlist, on test

The §4.5a decomposition repeated on test. The fallback stratum returns Arm 1's top-1
unchanged, so any net loss comes from the items the LLM actively re-ranked.


In [ ]:
if "arm3_llm_rerank" in frames:
    decomposition = results.rerank_decomposition(frames["arm3_llm_rerank"])
    decomposition.to_csv(ARTEFACTS_DIR / "arm3_test_decomposition.csv", index=False)
    display(decomposition)
else:
    print("Arm 3 test predictions absent.")

### §5.7d Worked examples

The brief asks for examples of correct and incorrect predictions per method. Sampled with the
project seed so the report's examples are stable across reruns.

In [ ]:
worked = results.worked_examples(frames, n=4)
worked.to_csv(ARTEFACTS_DIR / "test_worked_examples.csv", index=False)
worked

### §5.7e Figure 8 — confusion pairs

*Report Figure 8.* One panel per arm. A bar is annotated only where the gold label had a
sibling to be confused with, so a pair that could never have been a fine-grained error is not
shown as one.


In [ ]:
fig, axes = plt.subplots(1, len(frames), figsize=(4.8 * len(frames), 3.8))
axes = axes if len(frames) > 1 else [axes]

for ax, (arm, frame) in zip(axes, frames.items()):
    pairs = results.confusion_pairs(frame, top_n=8, label_space=label_space)
    labels = [f"{g[:24]} -> {p[:24]}" for g, p in zip(pairs["gold"], pairs["pred"])]
    colours = ["#a8422f" if poss else "#4c72b0" for poss in pairs["family_error_possible"]]
    y = range(len(pairs))
    ax.barh(list(y), pairs["n"], color=colours, height=0.6)
    ax.set_yticks(list(y))
    ax.set_yticklabels(labels, fontsize=6)
    ax.invert_yaxis()
    ax.set_xlabel("errors")
    ax.set_title(results.ARM_LABELS.get(arm, arm), fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.get_major_locator().set_params(integer=True)

fig.suptitle("Most frequent test-set confusions, one panel per arm", fontsize=10)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig8_confusions.png", dpi=200, bbox_inches="tight")
plt.show()

### §5.7f Figure 9 — test accuracy with bootstrap CIs

*Report Figure 9.* Error bars are the percentile bootstrap 95% CIs from §5.5, not ±1 SE, so the
visual comparison matches the interval quoted in the text.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))

order = test_scores.sort_values("accuracy", ascending=True)
y = range(len(order))
lower = order["accuracy"] - order["ci_low"]
upper = order["ci_high"] - order["accuracy"]

ax.barh(list(y), order["accuracy"], color="#4c72b0", height=0.55)
ax.errorbar(order["accuracy"], list(y), xerr=[lower, upper], fmt="none",
            ecolor="#22303f", capsize=4)
ax.set_yticks(list(y))
ax.set_yticklabels(order["label"])
ax.set_xlabel("Test accuracy (200 items, bootstrap 95% CI)")
ax.set_xlim(0, 1)
for i, value, hi in zip(y, order["accuracy"], order["ci_high"]):
    ax.text(hi + 0.025, i, f"{value:.3f}", va="center", fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig9_test_accuracy.png", dpi=200)
plt.show()

mark("5. test run")

# §6 Workflow diagrams, inventory and export

*Backs report Figures 4 and 5.*

The diagrams are drawn by `amlh.diagrams`, which reads each hyperparameter it prints from
`config.HYPERPARAMETERS` rather than from hard-coded text, so a diagram cannot fall out of
step with the frozen configuration.


In [ ]:
from amlh import diagrams

diagrams.draw_arm1_workflow()
diagrams.draw_arms23_workflow()
plt.show()

## §6.1 Everything this notebook wrote

In [ ]:
artefact_names = sorted(p.name for p in ARTEFACTS_DIR.glob("*.csv"))
figure_names = sorted(p.name for p in FIGURES_DIR.glob("*.png"))

print(f"artefacts/ ({len(artefact_names)} CSVs)")
for name in artefact_names:
    print(f"  {name}")
print(f"\nfigures/ ({len(figure_names)} PNGs)")
for name in figure_names:
    print(f"  {name}")

## §6.2 Wall clock

Runtime of this run, by section. These are the figures quoted in the report's appendix note.

In [ ]:
print(f"{'section':28s} {'cumulative (min)':>18s} {'section (min)':>15s}")
previous = 0.0
for name, elapsed in SECTION_TIMES.items():
    print(f"{name:28s} {elapsed / 60:18.1f} {(elapsed - previous) / 60:15.1f}")
    previous = elapsed
print(f"\ntotal: {(time.perf_counter() - NOTEBOOK_START) / 60:.1f} min "
      f"| RUN_FULL_SELECTION = {RUN_FULL_SELECTION}")

## §6.3 Export

Bundles the artefacts and figures this run produced, so the outputs can be inspected outside the
runtime.

In [ ]:
export = Path("amlh_outputs.zip")
with zipfile.ZipFile(export, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder, prefix in ((ARTEFACTS_DIR, "artefacts"), (FIGURES_DIR, "figures")):
        for path in sorted(folder.glob("*")):
            if path.is_file():
                zf.write(path, arcname=f"{prefix}/{path.name}")

print(f"wrote {export.name} ({export.stat().st_size / 1e6:.1f} MB, "
      f"{len(zipfile.ZipFile(export).namelist())} members)")

if IN_COLAB:
    from google.colab import files
    files.download(str(export))